# 08 — Adaptive evidence acquisition (E9)

**Objective.** B1 reads ColQwen2's top 4 pages once. About 40 % of answerable development questions still lack part of their annotated evidence in those 4 pages. E9 asks whether a *retrieval-uncertainty trigger* can tell, before or after the first read, when evidence is missing, so that more pages are read only when needed. Pre-registered in `experiments/specs/e9_adaptive_evidence_v1.json` (commit `7f41eea`), proposal `docs/e9_proposal.md`.

**Stages.**
- **Stage 0 (this section, offline, $0).** Do any of five pre-registered signals predict "annotated evidence not fully in the top 4"? A gate with five conditions (G1–G5) decides whether any paid stage runs.
- Stage 1 (smoke) and Stage 2 (paid arms B1s, F8a, F8b, derived A1) are added below only if Stage 0 passes or its pre-registered fallback rule fires.

**Inputs** (all frozen, checksum-verified; no question text, no test-pool row is read):
- `experiments/frozen/e5/tables/rankings.parquet`: ColQwen2's full page ranking and MaxSim scores for the 118 held-out questions;
- `experiments/frozen/e5/tables/per_question.csv`: annotated evidence pages;
- `experiments/frozen/e8/tables/evaluation_per_question.csv`: B1's frozen reply properties (abstention, numeric support).

**Outputs:** `data/processed/mmlongbench_doc/e9/stage0_per_question.csv` and `stage0_summary.json` (no question, answer or page text).

**Mode.** `VISUALDOC_E9_MODE` defaults to `offline`. Nothing in Stage 0 creates an API client.

**Next step.** The Stage 0 verdict decides between Stage 1/2 and closing E9 at $0 (spec `after_stage0`).

## Method

**Target.** For a question with annotated evidence pages $E$ and ColQwen2 ranking $r_1, r_2, \dots$:

- $Y_{\text{incomplete}} = 1$ if $E \not\subseteq \{r_1,\dots,r_4\}$ — B1 never saw some evidence page (primary);
- $Y_{\text{recoverable}} = 1$ if $Y_{\text{incomplete}}$ and some missing page is at rank 5–8 — reading 8 pages could help (secondary).

**Signals** (higher = more uncertain; $s_1 \ge s_2 \ge \dots$ are MaxSim scores, $\mu,\sigma$ the mean and s.d. over all pages of the document):

| | Signal | Formula | Idea |
|---|---|---|---|
| U1 | margin 4–5 | $-(s_4 - s_5)/s_1$ | the cut after page 4 is arbitrary |
| U2 | top stand-out | $-(s_1-\mu)/\sigma$ | nothing in the document matches well |
| U3 | flatness 1–8 | $-(s_1 - s_8)/s_1$ | no clear winner among the top 8 |
| U4 | abstained | B1 replied `Not answerable` | the reader did not find it |
| U5 | numeric unsupported | B1's number not found on its cited page | the reader may have guessed |

U1–U3 are scale-free because a MaxSim score $\sum_{q}\max_{p}\langle e_q, e_p\rangle$ grows with the number of query tokens; dividing by $s_1$ or standardising by the document's own scores removes that.

**Why AUROC.** A trigger is useful if it ranks incomplete-evidence questions above complete ones. AUROC is that probability:
$$\text{AUROC} = P(U^+ > U^-) + \tfrac12 P(U^+ = U^-) = \frac{R^+ - n^+(n^+ + 1)/2}{n^+ n^-},$$
where $R^+$ is the sum of the mid-ranks of the positive questions. Chance = 0.5.

**Uncertainty.** Questions within a document are correlated (same layout, same retrieval difficulty), so the 14 **documents** are resampled (cluster bootstrap, 10,000 draws). One-sided p for "AUROC ≤ 0.5", Holm-adjusted over the five signals.

**Operating point.** A continuous signal triggers on the 40 % most uncertain questions of all 118 (its 60th percentile; label-free). Lift = P(incomplete | trigger) − base rate.

**Gate** (frozen; a signal passes only if all hold): G1 AUROC ≥ 0.65 · G2 Holm p ≤ 0.05 · G3 lift ≥ +0.15 with bootstrap lower bound > 0 · G4 every leave-one-document-out AUROC ≥ 0.60 · G5 at least 6 recoverable questions triggered.

**Assumptions and limits.** Evidence annotations are single-annotator and an unannotated page that also answers counts as missing. The 14 documents are development data that already shaped B1. U4/U5 come from one frozen B1 replicate.

In [1]:
# Setup: paths, the frozen spec, and the mode. Stage 0 is offline whatever the mode.
from pathlib import Path
import hashlib
import json
import os
import subprocess

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.width", 170)


def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not find the repository root")


PROJECT_ROOT = find_project_root()
FROZEN_DIR = PROJECT_ROOT / "experiments" / "frozen"
SPEC_PATH = PROJECT_ROOT / "experiments" / "specs" / "e9_adaptive_evidence_v1.json"
SPEC_FREEZE_COMMIT = "7f41eea"
OUT_DIR = PROJECT_ROOT / "data" / "processed" / "mmlongbench_doc" / "e9"

RANKINGS_PATH = FROZEN_DIR / "e5" / "tables" / "rankings.parquet"
EVIDENCE_PATH = FROZEN_DIR / "e5" / "tables" / "per_question.csv"
E8_TABLE_PATH = FROZEN_DIR / "e8" / "tables" / "evaluation_per_question.csv"

REQUIRED_INPUTS = {
    SPEC_PATH: "the E9 specification is tracked in git: restore it",
    RANKINGS_PATH: "restore experiments/frozen/e5 from git",
    EVIDENCE_PATH: "restore experiments/frozen/e5 from git",
    E8_TABLE_PATH: "restore experiments/frozen/e8 from git",
}
missing = [f"{path.relative_to(PROJECT_ROOT)} ({fix})" for path, fix in REQUIRED_INPUTS.items() if not path.is_file()]
assert not missing, "Notebook 08 inputs missing:\n  " + "\n  ".join(missing)

MODE = os.environ.get("VISUALDOC_E9_MODE", "offline")
assert MODE in {"offline", "smoke", "full"}, f"unknown VISUALDOC_E9_MODE {MODE!r}"
SPEC = json.loads(SPEC_PATH.read_text())
STAGE0 = SPEC["stage0"]
SEED, DRAWS = 20261126, 10_000            # spec stage0.statistics.bootstrap
TOP_K, DEEP_K = 4, 8                      # B1 reads 4 pages; the deeper read is 8
TRIGGER_QUANTILE = 0.60                   # trigger on the 40 % most uncertain questions
GATE = dict(auroc_min=0.65, alpha=0.05, lift_min=0.15, lodo_min=0.60, recoverable_min=6)
FALLBACK_RECOVERABLE_MIN = 12             # spec after_stage0.if_fail
NOT_ANSWERABLE = "Not answerable"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT.name}. Mode: {MODE} (Stage 0 never calls an API). Outputs: {OUT_DIR.relative_to(PROJECT_ROOT)}")

Project root: Multimodal-Document-Intelligence. Mode: offline (Stage 0 never calls an API). Outputs: data/processed/mmlongbench_doc/e9


In [2]:
# Integrity: the frozen packages equal their SHA256SUMS, and the spec is byte-identical to its pre-registration commit.
def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()


def verify_sha256sums(folder):
    entries = [line.split(maxsplit=1) for line in (folder / "SHA256SUMS").read_text().splitlines() if line.strip()]
    return {name: (folder / name).is_file() and sha256_bytes((folder / name).read_bytes()) == digest for digest, name in entries}


for name in ("e5", "e8"):
    checks = verify_sha256sums(FROZEN_DIR / name)
    assert checks and all(checks.values()), f"experiments/frozen/{name} differs from its SHA256SUMS"
    print(f"experiments/frozen/{name}: {len(checks)} files equal their SHA256SUMS")

committed_spec = subprocess.run(["git", "show", f"{SPEC_FREEZE_COMMIT}:experiments/specs/e9_adaptive_evidence_v1.json"],
                                cwd=PROJECT_ROOT, capture_output=True, check=True).stdout
assert committed_spec == SPEC_PATH.read_bytes(), "the E9 spec changed after its freeze commit: that is a new condition"
SPEC_SHA256 = sha256_bytes(committed_spec)
print(f"E9 spec equals commit {SPEC_FREEZE_COMMIT} (sha256 {SPEC_SHA256[:12]}…)")

experiments/frozen/e5: 11 files equal their SHA256SUMS
experiments/frozen/e8: 10 files equal their SHA256SUMS
E9 spec equals commit 7f41eea (sha256 c2cb8ef9f4ea…)


In [3]:
# Populations. P_deploy = all 118 held-out questions (label-free); P0 = the 101 with annotated evidence.
rankings = pd.read_parquet(RANKINGS_PATH)
v = rankings[(rankings["arm"] == "V") & (rankings["population"] == "held-out")].set_index("qid")
v = v.assign(pages=v["pages"].map(lambda p: [int(x) for x in p]), scores=v["scores"].map(lambda s: np.asarray(s, dtype=float)))
assert len(v) == 118 and v["doc_id"].nunique() == 14
assert all(np.all(np.diff(s) <= 0) for s in v["scores"]), "scores must be sorted best first"
assert v["pages"].map(len).min() >= DEEP_K

evidence = pd.read_csv(EVIDENCE_PATH)
evidence = evidence[evidence["population"] == "held-out"].set_index("qid")
evidence["evidence_pages"] = evidence["evidence_pages"].map(lambda s: [int(x) for x in json.loads(s)])
assert len(evidence) == 101 and (evidence["n_evidence"] >= 1).all() and set(evidence.index) <= set(v.index)

e8 = pd.read_csv(E8_TABLE_PATH)
b1 = e8[e8["arm"] == "L-B1"].set_index("qid")
assert len(b1) == 118 and set(b1.index) == set(v.index) and int(b1["answerable"].sum()) == 102

q = v[["doc_id", "pages", "scores"]].join(b1[["answerable", "abstained", "numeric_support"]])
q["labelled"] = q.index.isin(evidence.index)
q["evidence_pages"] = evidence["evidence_pages"].reindex(q.index)
print(f"P_deploy: {len(q)} questions in {q['doc_id'].nunique()} documents ({int(q['answerable'].sum())} answerable). "
      f"P0 (annotated evidence): {int(q['labelled'].sum())} questions in {q.loc[q['labelled'], 'doc_id'].nunique()} documents.")
print("Answerable without annotation (not in P0):", sorted(q.index[q["answerable"] & ~q["labelled"]]))

P_deploy: 118 questions in 14 documents (102 answerable). P0 (annotated evidence): 101 questions in 14 documents.
Answerable without annotation (not in P0): ['mmlb-0599']


In [4]:
# Targets, shown once on one question, then computed for all of P0.
def evidence_targets(evidence_pages, ranked_pages):
    top4, ranks5_8 = set(ranked_pages[:TOP_K]), set(ranked_pages[TOP_K:DEEP_K])
    missing = set(evidence_pages) - top4
    return {
        "incomplete": bool(missing),
        "recoverable": bool(missing & ranks5_8),
        "complete_at_8": set(evidence_pages) <= set(ranked_pages[:DEEP_K]),
        "any_at_4": bool(set(evidence_pages) & top4),
        "n_evidence": len(evidence_pages),
    }

example = q[q["labelled"]].index[0]
print(f"{example}: evidence {q.at[example, 'evidence_pages']}, top 8 {q.at[example, 'pages'][:DEEP_K]} ->",
      evidence_targets(q.at[example, "evidence_pages"], q.at[example, "pages"]))

targets = pd.DataFrame({qid: evidence_targets(row.evidence_pages, row.pages) for qid, row in q[q["labelled"]].iterrows()}).T
TARGET_TYPES = {"incomplete": bool, "recoverable": bool, "complete_at_8": bool, "any_at_4": bool, "n_evidence": int}
q = q.join(targets.astype(TARGET_TYPES))  # NaN for the 17 questions without annotation (not in P0)


def labelled_view(frame):
    """P0 with the target columns typed (the join above leaves them object-typed on all 118 rows)."""
    return frame[frame["labelled"]].astype(TARGET_TYPES)


P0 = labelled_view(q)
base_rate = P0["incomplete"].mean()
print(f"P0: incomplete at k=4 {int(P0['incomplete'].sum())}/{len(P0)} (base rate {base_rate:.3f}); "
      f"recoverable at ranks 5-8 {int(P0['recoverable'].sum())}; complete at k=8 but not k=4 "
      f"{int((P0['complete_at_8'] & P0['incomplete']).sum())}; no evidence at all in the top 4 {int((~P0['any_at_4']).sum())}.")

mmlb-0042: evidence [12], top 8 [12, 13, 35, 56, 11, 41, 37, 59] -> {'incomplete': False, 'recoverable': False, 'complete_at_8': True, 'any_at_4': True, 'n_evidence': 1}
P0: incomplete at k=4 39/101 (base rate 0.386); recoverable at ranks 5-8 15; complete at k=8 but not k=4 8; no evidence at all in the top 4 15.


In [5]:
# The five pre-registered signals (higher = more uncertain). The arithmetic is shown once, then applied to all 118.
s = q.at[example, "scores"]
print(f"{example}: s1..s8 = {np.round(s[:8], 3).tolist()}; mean {s.mean():.3f}, sd {s.std():.3f}")
print(f"  U1 = -({s[3]:.3f} - {s[4]:.3f}) / {s[0]:.3f} = {-(s[3] - s[4]) / s[0]:.4f}")
print(f"  U2 = -({s[0]:.3f} - {s.mean():.3f}) / {s.std():.3f} = {-(s[0] - s.mean()) / s.std():.4f}")
print(f"  U3 = -({s[0]:.3f} - {s[7]:.3f}) / {s[0]:.3f} = {-(s[0] - s[7]) / s[0]:.4f}")

SIGNALS = ["U1_margin_45", "U2_top_standout", "U3_flatness_18", "U4_abstained", "U5_numeric_unsupported"]
CONTINUOUS = SIGNALS[:3]
q["U1_margin_45"] = q["scores"].map(lambda s: -(s[3] - s[4]) / s[0])
q["U2_top_standout"] = q["scores"].map(lambda s: -(s[0] - s.mean()) / s.std(ddof=0))
q["U3_flatness_18"] = q["scores"].map(lambda s: -(s[0] - s[7]) / s[0])
q["U4_abstained"] = q["abstained"].astype(bool).astype(int)
q["U5_numeric_unsupported"] = q["numeric_support"].map(lambda x: int(str(x) == "False"))
assert q[SIGNALS].notna().all().all()

thresholds = {name: float(np.quantile(q[name], TRIGGER_QUANTILE)) for name in CONTINUOUS}  # over P_deploy, no label used
thresholds.update({name: 1.0 for name in SIGNALS[3:]})
for name in SIGNALS:
    q[f"trigger_{name}"] = q[name] >= thresholds[name]
P0 = labelled_view(q)
display(pd.DataFrame({"threshold": thresholds, "trigger rate (118)": {n: q[f"trigger_{n}"].mean() for n in SIGNALS},
                      "trigger rate (P0)": {n: P0[f"trigger_{n}"].mean() for n in SIGNALS}}).round(4))

mmlb-0042: s1..s8 = [23.62, 17.616, 17.226, 17.172, 16.796, 16.302, 15.997, 15.87]; mean 12.563, sd 2.739
  U1 = -(17.172 - 16.796) / 23.620 = -0.0159
  U2 = -(23.620 - 12.563) / 2.739 = -4.0376
  U3 = -(23.620 - 15.870) / 23.620 = -0.3281


,threshold,trigger rate (118),trigger rate (P0)
U1_margin_45,-0.0066,0.3983,0.4059
U2_top_standout,-2.4228,0.3983,0.4059
U3_flatness_18,-0.1531,0.3983,0.3960
U4_abstained,1.0000,0.3051,0.2574
U5_numeric_unsupported,1.0000,0.2373,0.2673


In [6]:
# Statistics helpers. AUROC by mid-ranks is checked against the brute-force pair count on random data with ties.
def midranks(x):
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty(len(x))
    sorted_x = x[order]
    i = 0
    while i < len(x):
        j = i
        while j + 1 < len(x) and sorted_x[j + 1] == sorted_x[i]:
            j += 1
        ranks[order[i:j + 1]] = (i + j) / 2 + 1
        i = j + 1
    return ranks


def auroc(score, y):
    score, y = np.asarray(score, dtype=float), np.asarray(y, dtype=bool)
    n_pos, n_neg = y.sum(), (~y).sum()
    if n_pos == 0 or n_neg == 0:
        return np.nan
    return (midranks(score)[y].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)


def auroc_bruteforce(score, y):
    pos, neg = score[y], score[~y]
    return ((pos[:, None] > neg[None, :]).sum() + 0.5 * (pos[:, None] == neg[None, :]).sum()) / (len(pos) * len(neg))


check_rng = np.random.default_rng(0)
for _ in range(200):
    toy_y = check_rng.random(40) < 0.4
    toy_s = check_rng.integers(0, 6, 40).astype(float)  # many ties on purpose
    if toy_y.any() and (~toy_y).any():
        assert abs(auroc(toy_s, toy_y) - auroc_bruteforce(toy_s, toy_y)) < 1e-12
print("AUROC (mid-ranks) equals the brute-force pair count on 200 random tied samples.")


def holm(pvalues):
    p = np.asarray(pvalues, dtype=float)
    order = np.argsort(p)
    adjusted = np.empty_like(p)
    running = 0.0
    for rank, index in enumerate(order):
        running = max(running, min(1.0, (len(p) - rank) * p[index]))
        adjusted[index] = running
    return adjusted


assert np.allclose(holm([0.01, 0.04, 0.03]), [0.03, 0.06, 0.06])

DOCS = sorted(P0["doc_id"].unique())
doc_rows = {doc: np.flatnonzero(P0["doc_id"].to_numpy() == doc) for doc in DOCS}
draw_rng = np.random.default_rng(SEED)
DRAW_DOCS = draw_rng.integers(0, len(DOCS), size=(DRAWS, len(DOCS)))  # the same document resamples for every signal
DRAW_ROWS = [np.concatenate([doc_rows[DOCS[d]] for d in draw]) for draw in DRAW_DOCS]
print(f"{DRAWS} document-cluster resamples of {len(DOCS)} documents drawn once (seed {SEED}).")

AUROC (mid-ranks) equals the brute-force pair count on 200 random tied samples.
10000 document-cluster resamples of 14 documents drawn once (seed 20261126).


In [7]:
# The Stage 0 gate, exactly as frozen: G1-G5 per signal, Holm over the five.
y = P0["incomplete"].to_numpy()
y_rec = P0["recoverable"].to_numpy()
rows = []
for name in SIGNALS:
    score = P0[name].to_numpy(dtype=float)
    trig = P0[f"trigger_{name}"].to_numpy()
    point = auroc(score, y)
    boot_auc, boot_lift, discarded = [], [], 0
    for rows_idx in DRAW_ROWS:
        yb, sb, tb = y[rows_idx], score[rows_idx], trig[rows_idx]
        if yb.all() or (~yb).all():
            discarded += 1
            continue
        boot_auc.append(auroc(sb, yb))
        if tb.any():
            boot_lift.append(yb[tb].mean() - yb.mean())
    boot_auc, boot_lift = np.array(boot_auc), np.array(boot_lift)
    lodo = [auroc(score[P0["doc_id"].to_numpy() != doc], y[P0["doc_id"].to_numpy() != doc]) for doc in DOCS]
    precision = y[trig].mean() if trig.any() else np.nan
    rows.append({
        "signal": name,
        "auroc": point,
        "auroc_ci95": tuple(np.round(np.percentile(boot_auc, [2.5, 97.5]), 3)),
        "p_one_sided": (1 + (boot_auc <= 0.5).sum()) / (1 + len(boot_auc)),
        "lodo_min": float(np.nanmin(lodo)),
        "triggered_P0": int(trig.sum()),
        "precision": precision,
        "lift": precision - base_rate,
        "lift_ci95": tuple(np.round(np.percentile(boot_lift, [2.5, 97.5]), 3)),
        "recoverable_triggered": int((trig & y_rec).sum()),
        "auroc_recoverable": auroc(score, y_rec),
        "draws_discarded": discarded,
    })
gate = pd.DataFrame(rows).set_index("signal")
gate["p_holm"] = holm(gate["p_one_sided"].to_numpy())
gate["G1"] = gate["auroc"] >= GATE["auroc_min"]
gate["G2"] = gate["p_holm"] <= GATE["alpha"]
gate["G3"] = (gate["lift"] >= GATE["lift_min"]) & gate["lift_ci95"].map(lambda ci: ci[0] > 0)
gate["G4"] = gate["lodo_min"] >= GATE["lodo_min"]
gate["G5"] = gate["recoverable_triggered"] >= GATE["recoverable_min"]
gate["passes"] = gate[["G1", "G2", "G3", "G4", "G5"]].all(axis=1)
display(gate.round(3))

passing = gate[gate["passes"]]
STAGE0_PASSED = bool(len(passing))
SELECTED = passing.sort_values("auroc", ascending=False, kind="mergesort").index[0] if STAGE0_PASSED else None
print(f"Base rate of incomplete evidence: {base_rate:.3f} ({int(y.sum())}/{len(y)}).")
print(f"STAGE 0 {'PASSED' if STAGE0_PASSED else 'FAILED'}: " +
      (f"selected trigger {SELECTED} (threshold {thresholds[SELECTED]:.6f})" if STAGE0_PASSED else "no signal satisfies G1-G5"))

,auroc,auroc_ci95,p_one_sided,lodo_min,triggered_P0,precision,lift,lift_ci95,recoverable_triggered,auroc_recoverable,draws_discarded,p_holm,G1,G2,G3,G4,G5,passes
signal,,,,,,,,,,,,,,,,,,
U1_margin_45,0.622,"(0.541, 0.701)",0.001,0.606,41,0.488,0.102,"(0.022, 0.181)",10,0.683,0,0.006,False,True,False,True,True,False
U2_top_standout,0.620,"(0.517, 0.719)",0.011,0.584,41,0.415,0.028,"(-0.062, 0.135)",6,0.543,0,0.023,False,True,False,False,True,False
U3_flatness_18,0.783,"(0.661, 0.861)",0.000,0.752,40,0.625,0.239,"(0.095, 0.362)",8,0.634,0,0.000,True,True,True,True,True,True
U4_abstained,0.645,"(0.54, 0.727)",0.001,0.595,26,0.654,0.268,"(0.077, 0.446)",6,0.584,0,0.006,False,True,True,False,True,False
U5_numeric_unsupported,0.470,"(0.396, 0.527)",0.839,0.438,27,0.333,-0.053,"(-0.196, 0.046)",3,0.460,0,0.839,False,False,False,False,False,False


Base rate of incomplete evidence: 0.386 (39/101).
STAGE 0 PASSED: selected trigger U3_flatness_18 (threshold -0.153079)


In [8]:
# Descriptive only (spec stage0.descriptive_only): strata, unanswerable trigger rates, k = 8 headroom, and the fallback rule.
P0["span"] = np.where(P0["n_evidence"] == 1, "single-page", "multi-page")
strata = pd.DataFrame({
    span: {**{"questions": len(g), "incomplete": int(g["incomplete"].sum())},
           **{name: auroc(g[name], g["incomplete"]) for name in SIGNALS}}
    for span, g in P0.groupby("span")}).T
display(strata.round(3))

unanswerable = q[~q["answerable"]]
print("Trigger rate on the 16 unanswerable questions:",
      {name: round(float(unanswerable[f"trigger_{name}"].mean()), 3) for name in SIGNALS})

per_doc = P0.groupby("doc_id").agg(questions=("incomplete", "size"), incomplete=("incomplete", "mean"),
                                    **{f"trig_{n[:2]}": (f"trigger_{n}", "mean") for n in SIGNALS})
display(per_doc.round(2))

headroom = {
    "incomplete_at_4": int(P0["incomplete"].sum()),
    "recoverable_ranks_5_8": int(P0["recoverable"].sum()),
    "complete_at_8_not_4": int((P0["complete_at_8"] & P0["incomplete"]).sum()),
    "no_evidence_in_top_8": int(P0.apply(lambda r: not set(r.evidence_pages) & set(r.pages[:DEEP_K]), axis=1).sum()),
}
FALLBACK_RUNS = (not STAGE0_PASSED) and headroom["recoverable_ranks_5_8"] >= FALLBACK_RECOVERABLE_MIN
print("k = 8 headroom on P0:", headroom)
if STAGE0_PASSED:
    print("Next (spec after_stage0.if_pass): Stage 1 smoke, then B1s, F8a, F8b and derived A1.")
else:
    print(f"Fallback rule: Y_recoverable = {headroom['recoverable_ranks_5_8']} "
          f"{'>=' if FALLBACK_RUNS else '<'} {FALLBACK_RECOVERABLE_MIN} -> "
          + ("fixed-depth arms B1s, F8a, F8b run (no A1)." if FALLBACK_RUNS else "E9 closes at $0; B1 keeps k = 4."))

,questions,incomplete,U1_margin_45,U2_top_standout,U3_flatness_18,U4_abstained,U5_numeric_unsupported
multi-page,46.0,33.0,0.688,0.522,0.739,0.635,0.429
single-page,55.0,6.0,0.704,0.626,0.823,0.679,0.451


Trigger rate on the 16 unanswerable questions: {'U1_margin_45': 0.312, 'U2_top_standout': 0.312, 'U3_flatness_18': 0.375, 'U4_abstained': 0.562, 'U5_numeric_unsupported': 0.062}


,questions,incomplete,trig_U1,trig_U2,trig_U3,trig_U4,trig_U5
doc_id,,,,,,,
11-21-16-Updated-Post-Election-Release.pdf,7,0.14,0.43,0.29,0.00,0.00,0.29
2021-Apple-Catalog.pdf,3,0.33,0.67,0.33,0.33,0.00,0.33
2310.05634v2.pdf,6,0.33,0.50,0.50,0.00,0.50,0.00
2312.10997v5.pdf,6,0.33,0.67,0.50,0.50,0.67,0.00
3276a5b991c49cf5f9a4af0f7d6fce67.pdf,6,0.17,0.17,0.67,0.50,0.00,0.17
AMAZON_2017_10K.pdf,15,0.60,0.47,0.20,0.60,0.53,0.27
COSTCO_2021_10K.pdf,16,0.56,0.44,0.50,0.75,0.19,0.38
Macbook_air.pdf,6,0.50,0.67,0.67,0.50,0.33,0.17
NUS-Business-School-BBA-Brochure-2024.pdf,5,0.20,0.20,0.20,0.20,0.20,0.40


k = 8 headroom on P0: {'incomplete_at_4': 39, 'recoverable_ranks_5_8': 15, 'complete_at_8_not_4': 8, 'no_evidence_in_top_8': 11}
Next (spec after_stage0.if_pass): Stage 1 smoke, then B1s, F8a, F8b and derived A1.


In [9]:
# Save the per-question table (identifiers, targets, signals, triggers; no text) and the summary.
columns = ["doc_id", "answerable", "labelled", "n_evidence", "incomplete", "recoverable", "complete_at_8", "any_at_4",
           *SIGNALS, *[f"trigger_{n}" for n in SIGNALS]]
q[columns].to_csv(OUT_DIR / "stage0_per_question.csv")

def plain(value):
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        return None if np.isnan(value) else round(float(value), 6)
    if isinstance(value, tuple):
        return [plain(v) for v in value]
    return value

summary = {
    "spec": {"path": "experiments/specs/e9_adaptive_evidence_v1.json", "freeze_commit": SPEC_FREEZE_COMMIT, "sha256": SPEC_SHA256},
    "population": {"P_deploy": len(q), "P0": len(P0), "documents": len(DOCS)},
    "base_rate_incomplete": round(float(base_rate), 6),
    "thresholds": {k: round(v, 6) for k, v in thresholds.items()},
    "gate": {name: {k: plain(v) for k, v in row.items()} for name, row in gate.to_dict(orient="index").items()},
    "stage0_passed": STAGE0_PASSED,
    "selected_trigger": SELECTED,
    "selected_threshold": None if SELECTED is None else round(thresholds[SELECTED], 6),
    "headroom": headroom,
    "fallback_runs": bool(FALLBACK_RUNS),
    "strata_auroc": {span: {k: plain(v) for k, v in row.items()} for span, row in strata.to_dict(orient="index").items()},
    "bootstrap": {"draws": DRAWS, "seed": SEED, "unit": "document"},
}
(OUT_DIR / "stage0_summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n")
print(f"Wrote {OUT_DIR.relative_to(PROJECT_ROOT)}/stage0_per_question.csv ({len(q)} rows) and stage0_summary.json")

Wrote data/processed/mmlongbench_doc/e9/stage0_per_question.csv (118 rows) and stage0_summary.json


## Validation

The checks above already assert the inputs' checksums, that the spec equals its freeze commit, the population sizes (118 / 101 / 14), that the ranking scores are sorted, that every document has at least 8 pages, and that the AUROC implementation equals a brute-force pair count. The cell below adds calibration checks of the *procedure* (none of them is part of the frozen gate):

1. **Failure case (null calibration).** If the evidence labels are permuted across all 101 questions, every real association is destroyed. AUROC must then centre on 0.5, and a signal must rarely look as strong as observed. A procedure that found signal in shuffled labels would be measuring noise.
2. **Within-document decomposition (descriptive).** Permuting labels only *within* each document keeps each document's base rate. Any AUROC above 0.5 under that null comes from *between-document* differences: some documents are both flatter and harder. A signal that beats this null also carries *within-document* information about which questions lack evidence.
3. **Known-answer case.** An oracle signal equal to the target must score AUROC 1 in every leave-one-document-out fold. This shows the gate *can* pass when there is something to find.

*Correction recorded on first execution (2026-11-25):* this cell first asserted that the within-document null centres on 0.5. It does not (U3: 0.61), for the reason given in point 2. The assertion was a wrong expectation about the null, not a property of the gate. The global permutation is the correct null calibration, and the within-document permutation is kept as a descriptive decomposition. The frozen gate (G1–G5) and its result are unchanged by this correction.

In [10]:
# Null calibration (global permutation), within-document decomposition, and a known-answer oracle. None is a project result.
perm_rng = np.random.default_rng(SEED + 1)
doc_array = P0["doc_id"].to_numpy()
PERMUTATIONS = 1000
global_null = {name: [] for name in CONTINUOUS}
within_null = {name: [] for name in CONTINUOUS}
for _ in range(PERMUTATIONS):
    y_global = perm_rng.permutation(y)
    y_within = y.copy()
    for doc in DOCS:
        idx = np.flatnonzero(doc_array == doc)
        y_within[idx] = perm_rng.permutation(y_within[idx])
    for name in CONTINUOUS:
        score = P0[name].to_numpy(dtype=float)
        global_null[name].append(auroc(score, y_global))
        within_null[name].append(auroc(score, y_within))


def perm_p(null, observed):
    return (1 + np.sum(np.asarray(null) >= observed)) / (1 + len(null))


null_table = pd.DataFrame({name: {
    "observed": gate.at[name, "auroc"],
    "global null mean": np.mean(global_null[name]), "global p": perm_p(global_null[name], gate.at[name, "auroc"]),
    "within-doc null mean": np.mean(within_null[name]), "within-doc null 95th": np.percentile(within_null[name], 95),
    "within-doc p": perm_p(within_null[name], gate.at[name, "auroc"]),
} for name in CONTINUOUS}).T
display(null_table.round(3))
assert all(abs(np.mean(v) - 0.5) < 0.02 for v in global_null.values()), "the global-permutation null must centre on 0.5"

oracle = y.astype(float)
assert auroc(oracle, y) == 1.0
assert min(auroc(oracle[doc_array != d], y[doc_array != d]) for d in DOCS) == 1.0
WITHIN_DOC_P = {name: float(null_table.at[name, "within-doc p"]) for name in CONTINUOUS}
print("Global null centres on AUROC 0.5; the oracle scores 1.0 in every leave-one-document-out fold.")
print("Within-document p (descriptive):", {k: round(v, 3) for k, v in WITHIN_DOC_P.items()})

,observed,global null mean,global p,within-doc null mean,within-doc null 95th,within-doc p
U1_margin_45,0.622,0.500,0.021,0.544,0.634,0.082
U2_top_standout,0.620,0.500,0.027,0.535,0.627,0.060
U3_flatness_18,0.783,0.501,0.001,0.604,0.683,0.001


Global null centres on AUROC 0.5; the oracle scores 1.0 in every leave-one-document-out fold.
Within-document p (descriptive): {'U1_margin_45': 0.082, 'U2_top_standout': 0.06, 'U3_flatness_18': 0.001}


## Stage 0 result and its independent audit (written before any paid call)

**Verdict under the frozen gate: Stage 0 passed.** U3 (flatness of the top-8 ColQwen2 scores) is the only signal that satisfies G1–G5:

| | U3 |
|---|---|
| AUROC | 0.783 [0.661, 0.861] |
| Holm-adjusted p | 0.0005 (the bootstrap floor 5/10,001: no draw had AUROC ≤ 0.5) |
| Lift at a 40 % trigger rate | +0.239 [+0.095, +0.362] |
| Minimum leave-one-document-out AUROC | 0.752 |
| Recoverable questions triggered | 8 |

Frozen threshold: −0.153079, which triggers on 47/118 questions. U5 (numeric support of the answer) is at chance (0.47). That is consistent with E8: the grounding check is a poor repair signal.

**Independent audit (2026-11-25).** A separate agent wrote its own script from the frozen files only, with pairwise-count AUROC. It reproduced every number (max difference ≤ 5e-7; bootstrap intervals to the printed digit) and found that all foundations hold:
- **Page numbering.** It matches E5's frozen `recall@4_V` on 101/101 questions. A one-page shift would create 47 mismatches, so the check can detect the error.
- **Pages read.** The rankings are the pages B1 read (236/236 requests).
- **Leakage.** No signal uses evidence labels or correctness.
- **Population.** No test-pool document is involved.
- **Threshold.** It is label-free.
- **Not a document effect only.** Without the two 10-Ks, AUROC is 0.724 [0.585, 0.841]. Within documents it is 0.763.
- **Not a proxy.** Page count gives AUROC 0.605 (Spearman 0.24 with U3).

**What the pass does *not* show — the audit's main finding, recorded here before Stage 2.**
- U3 predicts *incomplete* evidence. It does **not** predict *recoverable-at-8*:
  - AUROC 0.634 against "evidence at ranks 5–8";
  - **0.480** against "complete at 8 but not at 4".
- At the operating point, the 40 triggered labelled questions break down as follows:
  - 8 recoverable;
  - 17 incomplete but not recoverable with 8 pages;
  - 15 already complete (wasted reads).
- Of the 8 questions that 8 pages would fully fix, it catches only 3.
- From the frozen E8 replies (descriptive), B1 is wrong on 7 of the 8 triggered recoverable questions, so **A1 − B1s cannot exceed about +0.069**. A realistic expectation is +0.03 to +0.04, below the +0.05 engineering criterion. `no_meaningful_gain` or `more_pages_not_triggering` are therefore the likely verdicts.
- G5 is a weak futility gate. A random 40 % trigger would pass it with probability 0.59.

None of this changes a frozen rule. Stage 2 runs as pre-registered because the owner authorised it, it is cheap (plan upper bound $0.38), and its fixed-depth arm answers a product question that no frozen result settles: whether `gpt-6-luna` should read 8 pages instead of B1's 4. That limit was inherited from GPT-4o-mini's image billing.

**Recorded deviations (no effect on any number).**
1. The spec names the answer-side input as `data/processed/.../b1/e8_evaluation_per_question.csv`. The notebook reads the checksum-protected frozen copy `experiments/frozen/e8/tables/evaluation_per_question.csv`. The audit confirmed that `abstained` and `numeric_support` are identical on all 354 rows; only `v2_final` differs (pending vs final labels).
2. The bootstrap percentile intervals rest on 14 clusters and may be slightly narrow. The within-document permutation test supports the same conclusion.

## Stages 1–2: the paid arms (authorised 2026-11-25, cumulative ceiling $0.50)

**Why these arms.** The selected trigger decides *when* to read 8 pages instead of 4. Two confounds must be separated from the trigger's value:

1. **More pages, not the trigger.** If reading 8 pages helps on *every* question, a trigger adds nothing. → **F8**: always 8 pages.
2. **Session drift and reader noise.** `gpt-6-luna` is an alias without a snapshot, and E8 measured 7 flips between identical replicates. Reusing E8's frozen replies as the control would mix "more pages" with "different day". → **B1s**: B1 re-run in the same session, request-identical to frozen E8 L-B1.

**A1 needs no extra call.** For a pre-read trigger (U3 uses only retrieval scores), the adaptive policy *is* "read 8 pages if triggered, else 4". Its reply on a triggered question is therefore the F8 reply, and on any other question the B1s reply:
$$A1_r(q) = \begin{cases} F8_r(q) & \text{if } U3(q) \ge \tau \\ B1s(q) & \text{otherwise}\end{cases}, \quad r \in \{a, b\}.$$
So $A1 - B1s$ is non-zero only on triggered questions, and $A1 - F8$ only on untriggered ones. That last difference is exactly what the trigger *saves*: the questions where 8 pages would have hurt.

**Order and replicates.** B1s and F8a are adjacent per question in a counterbalanced order (59/59, seed 20261127). The F8b block follows all pairs, and is never served from a cache. 354 calls.

**Fail-closed execution** (as E8): append-only log; no SDK retries; at most 2 attempts per index; stop on any API error, contract failure, or change of served model or system fingerprint; cumulative observed cost (unknown usage charged at its upper bound) can never exceed $0.50 across smoke and full.

**What this section does offline.** It rebuilds every request without the API, proves B1s equals frozen E8 L-B1 byte for byte (fingerprints), proves the token model reproduces E8's billed prompt tokens, writes the plan, and mock-tests the executor. A client is created only in `smoke` or `full` mode, after a readiness gate.

In [11]:
# Rebuild every Stage 2 request visibly from the frozen E8 log and the frozen 768-px render manifest (no src/visualdoc import).
import base64
from collections import Counter
from datetime import datetime, timezone
from math import ceil, floor
from types import SimpleNamespace
import tempfile
import time
import uuid

import tiktoken

assert STAGE0_PASSED or FALLBACK_RUNS, "Stage 0 failed and the fallback rule did not fire: E9 has no paid stage"
A1_EXISTS = STAGE0_PASSED
E8_RUN = FROZEN_DIR / "e8" / "run_20261125T140433934010Z_4edb90e7_full"
manifest = pd.read_csv(FROZEN_DIR / "e5" / "tables" / "render_manifest_heldout.csv").set_index(["doc_id", "page"])


def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]


e8_log = read_jsonl(E8_RUN / "execution_log.jsonl")
e8_logged = {(record["qid"], record["arm"]): record for record in e8_log}
assert len(e8_logged) == 354


def png_size(png):
    assert png.startswith(b"\x89PNG\r\n\x1a\n") and len(png) >= 24, "not a PNG"
    return int.from_bytes(png[16:20], "big"), int.from_bytes(png[20:24], "big")


def page_png(doc_id, page):
    """The frozen 768-px render of one page, refused if its bytes changed since E5."""
    row = manifest.loc[(doc_id, int(page))]
    png = (PROJECT_ROOT / row["image_path"]).read_bytes()
    assert sha256_bytes(png) == row["sha256"], f"stale render: {doc_id} p.{page}"
    return png


def build_request(template, doc_id, pages):
    """B1's layout: system prompt; 'Question: ...\n\nPage images:'; then '[Page N]' and the page image, best-ranked first."""
    header = template["messages"][1]["content"][0]
    assert header["type"] == "text" and header["text"].startswith("Question: ")
    content = [dict(header)]
    for page in pages:
        png = page_png(doc_id, page)
        content.append({"type": "text", "text": f"[Page {page}]"})
        content.append({"type": "image_url", "image_url": {"url": "data:image/png;base64," + base64.b64encode(png).decode("ascii"),
                                                           "detail": "high"}})
    return {**{key: value for key, value in template.items() if key != "messages"},
            "messages": [dict(template["messages"][0]), {"role": "user", "content": content}]}


def loggable(request):
    """The request with each image replaced by its SHA-256, byte length, dimensions and detail: safe to log, no Base64."""
    messages = []
    for message in request["messages"]:
        if isinstance(message["content"], str):
            messages.append(dict(message))
            continue
        parts = []
        for part in message["content"]:
            if part["type"] == "image_url":
                png = base64.b64decode(part["image_url"]["url"].split(",", 1)[1])
                parts.append({"type": "image", "sha256": sha256_bytes(png), "bytes": len(png),
                              "dimensions": list(png_size(png)), "detail": part["image_url"].get("detail")})
            else:
                parts.append(part)
        messages.append({"role": message["role"], "content": parts})
    return {**{key: value for key, value in request.items() if key != "messages"}, "messages": messages}


def fingerprint(request):
    return sha256_bytes(json.dumps(loggable(request), sort_keys=True).encode())


# Shown once: B1s for one question equals the frozen E8 L-B1 request, and F8 extends it with pages 5-8.
example_template = e8_logged[(example, "L-B1")]["request"]
example_b1 = build_request(example_template, q.at[example, "doc_id"], q.at[example, "pages"][:TOP_K])
assert loggable(example_b1) == example_template and fingerprint(example_b1) == e8_logged[(example, "L-B1")]["fingerprint"]
print(f"{example}: rebuilt B1 request equals frozen E8 L-B1 (fingerprint {fingerprint(example_b1)[:12]}…); "
      f"F8 adds pages {q.at[example, 'pages'][TOP_K:DEEP_K]}.")
del example_b1

# All 118 questions. Requests are summarised (loggable form) and rebuilt just before sending, to keep memory small.
REQUESTS = {}
for qid in sorted(q.index):
    template = e8_logged[(qid, "L-B1")]["request"]
    doc_id, ranked = q.at[qid, "doc_id"], q.at[qid, "pages"]
    assert e8_logged[(qid, "L-B1r")]["fingerprint"] == e8_logged[(qid, "L-B1")]["fingerprint"]
    for arm, pages in (("B1s", ranked[:TOP_K]), ("F8", ranked[:DEEP_K])):
        request = build_request(template, doc_id, pages)
        REQUESTS[(qid, arm)] = {"doc_id": doc_id, "pages": [int(p) for p in pages], "loggable": loggable(request),
                                "fingerprint": fingerprint(request)}
    assert REQUESTS[(qid, "B1s")]["loggable"] == template, f"{qid}: B1s differs from frozen E8 L-B1"
    assert REQUESTS[(qid, "B1s")]["fingerprint"] == e8_logged[(qid, "L-B1")]["fingerprint"]
    assert REQUESTS[(qid, "F8")]["pages"][:TOP_K] == REQUESTS[(qid, "B1s")]["pages"]
    del request


def rebuild(qid, arm):
    """The full request (with Base64 images) for one planned call; refused unless it equals the planned fingerprint."""
    spec = REQUESTS[(qid, arm)]
    request = build_request(e8_logged[(qid, "L-B1")]["request"], spec["doc_id"], spec["pages"])
    assert fingerprint(request) == spec["fingerprint"], f"{qid} {arm}: request changed after planning"
    return request


print(f"Rebuilt {len(REQUESTS)} requests: 118 B1s equal the frozen E8 L-B1 requests byte for byte (fingerprints and loggable form); "
      "118 F8 requests extend them with ColQwen2 pages 5-8.")

mmlb-0042: rebuilt B1 request equals frozen E8 L-B1 (fingerprint 3c50abe7f924…); F8 adds pages [11, 41, 37, 59].


Rebuilt 236 requests: 118 B1s equal the frozen E8 L-B1 requests byte for byte (fingerprints and loggable form); 118 F8 requests extend them with ColQwen2 pages 5-8.


In [12]:
# The measured gpt-6-luna token model (frozen E8 smoke): o200k_base text + 9 multimodal framing + 100 strict schema
# + floor(1.2 * ceil(w/32) * ceil(h/32)) per image. It must reproduce every billed E8 prompt total exactly.
MODEL = "gpt-6-luna"
PRICE = {"input": 0.10 / 1e6, "cached_input": 0.01 / 1e6, "output": 0.50 / 1e6}
CONTEXT_TOKENS = 128_000
ENCODING = tiktoken.get_encoding("o200k_base")


def luna_image_tokens(width, height):
    return floor(1.2 * ceil(width / 32) * ceil(height / 32))


def prompt_tokens(logged_request):
    texts, dims = [], []
    for message in logged_request["messages"]:
        parts = [{"type": "text", "text": message["content"]}] if isinstance(message["content"], str) else message["content"]
        for part in parts:
            if part["type"] == "text":
                texts.append(part["text"])
            elif part["type"] == "image":
                dims.append(tuple(part["dimensions"]))
            else:
                raise ValueError(f"cannot count a {part['type']!r} part")
    framing = 9 if dims else 10
    schema = 100 if logged_request["response_format"]["type"] == "json_schema" else 0
    return sum(len(ENCODING.encode(text)) for text in texts) + framing + schema + sum(luna_image_tokens(*d) for d in dims)


def upper_cost(logged_request):
    return prompt_tokens(logged_request) * PRICE["input"] + logged_request["max_completion_tokens"] * PRICE["output"]


billed = [(record["prompt_tokens"], prompt_tokens(record["request"])) for record in e8_log if record["arm"] in ("L-B1", "L-B1r")]
assert len(billed) == 236 and all(observed == estimated for observed, estimated in billed), "token model no longer reproduces E8"
for spec in REQUESTS.values():
    spec["prompt_tokens"] = prompt_tokens(spec["loggable"])
    spec["upper_usd"] = upper_cost(spec["loggable"])
    assert spec["prompt_tokens"] + spec["loggable"]["max_completion_tokens"] <= CONTEXT_TOKENS
token_table = pd.DataFrame([{"arm": arm, "prompt_tokens": s["prompt_tokens"]} for (qid, arm), s in REQUESTS.items()])
display(token_table.groupby("arm")["prompt_tokens"].describe().round(0))
print("The token model reproduces all 236 billed E8 L-B1/L-B1r prompt totals exactly; every request fits the 128k context.")

,count,mean,std,min,25%,50%,75%,max
arm,,,,,,,,
B1s,118.0,3977.0,230.0,3097.0,4011.0,4016.0,4022.0,4259.0
F8,118.0,7641.0,461.0,5881.0,7715.0,7720.0,7726.0,8195.0


The token model reproduces all 236 billed E8 L-B1/L-B1r prompt totals exactly; every request fits the 128k context.


In [13]:
# The smoke plan (6 F8 calls, not scored) and the full plan (354 calls), both seeded; their bounds against the frozen ceilings.
SMOKE_SEED, ORDER_SEED = 20261129, 20261127
BUDGET_USD, SMOKE_BUDGET_USD, PLAN_UPPER_MAX_USD = 0.50, 0.02, 0.45
QIDS = sorted(q.index)

smoke_qids = sorted(np.random.default_rng(SMOKE_SEED).choice(QIDS, size=6, replace=False).tolist())
SMOKE_PLAN = [{"execution_index": i + 1, "replicate_id": "smoke", "qid": qid, "arm": "F8",
               "fingerprint": REQUESTS[(qid, "F8")]["fingerprint"], "prompt_tokens": REQUESTS[(qid, "F8")]["prompt_tokens"],
               "max_completion_tokens": 800} for i, qid in enumerate(smoke_qids)]

order = list(QIDS)
order_rng = np.random.default_rng(ORDER_SEED)
order_rng.shuffle(order)
b1s_first = set(order_rng.choice(order, size=59, replace=False).tolist())
FULL_PLAN = []
for pair_index, qid in enumerate(order, start=1):
    arms = ("B1s", "F8a") if qid in b1s_first else ("F8a", "B1s")
    for position, arm in enumerate(arms, start=1):
        request_arm = "B1s" if arm == "B1s" else "F8"
        FULL_PLAN.append({"execution_index": len(FULL_PLAN) + 1, "replicate_id": "main_r1", "pair_index": pair_index,
                          "position_in_pair": position, "qid": qid, "arm": arm,
                          "fingerprint": REQUESTS[(qid, request_arm)]["fingerprint"],
                          "prompt_tokens": REQUESTS[(qid, request_arm)]["prompt_tokens"], "max_completion_tokens": 800})
for qid in order:
    FULL_PLAN.append({"execution_index": len(FULL_PLAN) + 1, "replicate_id": "repeat_r1", "pair_index": None,
                      "position_in_pair": None, "qid": qid, "arm": "F8b", "fingerprint": REQUESTS[(qid, "F8")]["fingerprint"],
                      "prompt_tokens": REQUESTS[(qid, "F8")]["prompt_tokens"], "max_completion_tokens": 800})


def plan_totals(plan):
    prompt = sum(item["prompt_tokens"] for item in plan)
    output = sum(item["max_completion_tokens"] for item in plan)
    return {"calls": len(plan), "prompt_tokens": prompt, "max_output_tokens": output,
            "upper_usd": round(prompt * PRICE["input"] + output * PRICE["output"], 6),
            "plan_sha256": sha256_bytes(json.dumps(plan, sort_keys=True).encode())}


smoke_totals, full_totals = plan_totals(SMOKE_PLAN), plan_totals(FULL_PLAN)
assert Counter(item["arm"] for item in FULL_PLAN) == {"B1s": 118, "F8a": 118, "F8b": 118}
assert sum(item["arm"] == "B1s" and item["position_in_pair"] == 1 for item in FULL_PLAN) == 59
assert smoke_totals["upper_usd"] <= SMOKE_BUDGET_USD, smoke_totals
assert full_totals["upper_usd"] <= PLAN_UPPER_MAX_USD, full_totals
assert smoke_totals["upper_usd"] + full_totals["upper_usd"] <= BUDGET_USD
display(pd.DataFrame({"smoke": smoke_totals, "full": full_totals}).drop("plan_sha256"))

triggered = {qid: bool(q.at[qid, f"trigger_{SELECTED}"]) for qid in QIDS} if A1_EXISTS else {}
DESIGN = {
    "spec": {"path": "experiments/specs/e9_adaptive_evidence_v1.json", "freeze_commit": SPEC_FREEZE_COMMIT, "sha256": SPEC_SHA256},
    "stage0": {"passed": STAGE0_PASSED, "fallback_runs": bool(FALLBACK_RUNS), "selected_trigger": SELECTED,
               "threshold": None if SELECTED is None else round(thresholds[SELECTED], 6),
               "triggered_questions": sorted(qid for qid, t in triggered.items() if t)},
    "reader": {"model": MODEL, "reasoning_effort": "none", "temperature": 0, "max_completion_tokens": 800,
               "contract": "b1 strict JSON schema", "image": "768-px renders from the frozen E5 manifest"},
    "requests": {"B1s_equals_frozen_E8_L-B1": 118, "F8_extends_B1s_with_ranks_5_8": 118},
    "seeds": {"smoke": SMOKE_SEED, "order": ORDER_SEED},
    "smoke": {**smoke_totals, "qids": smoke_qids, "ceiling_usd": SMOKE_BUDGET_USD},
    "full": {**full_totals, "plan_upper_max_usd": PLAN_UPPER_MAX_USD},
    "budget": {"cumulative_ceiling_usd": BUDGET_USD, "sdk_retries": 0, "max_attempts_per_index": 2,
               "unknown_usage_charge": "the request's precomputed upper bound"},
}
(OUT_DIR / "stage2_design.json").write_text(json.dumps({**DESIGN, "smoke_plan": SMOKE_PLAN, "full_plan": FULL_PLAN}, indent=2) + "\n")
ADDENDUM_PATH = PROJECT_ROOT / "experiments" / "specs" / "e9_adaptive_evidence_v1_addendum.json"
ADDENDUM_FROZEN = ADDENDUM_PATH.is_file() and json.loads(ADDENDUM_PATH.read_text()) == json.loads(json.dumps(DESIGN))
print(f"A1 derivable: {A1_EXISTS}; triggered questions: {len(DESIGN['stage0']['triggered_questions'])}/118. "
      f"Technical addendum {'frozen and equal to this design' if ADDENDUM_FROZEN else 'NOT frozen yet (paid modes stay closed)'}.")

,smoke,full
calls,6,354
prompt_tokens,44472,2272656
max_output_tokens,4800,283200
upper_usd,0.006847,0.368866


A1 derivable: True; triggered questions: 47/118. Technical addendum frozen and equal to this design.


In [14]:
# The fail-closed executor (E8's rules, rebuilt visibly) and the B1 reply parser, checked against all 236 frozen E8 B1 replies.
def page_number(value):
    """A page >= 1: a JSON integer, a whole float (2.0), or ASCII digits ("4"). Never rounded."""
    if isinstance(value, int) and not isinstance(value, bool):
        number = value
    elif isinstance(value, float) and value.is_integer():
        number = int(value)
    elif isinstance(value, str) and value.isascii() and value.isdigit():
        number = int(value)
    else:
        raise ValueError(f"cited page {value!r} is not a page number")
    if number < 1:
        raise ValueError(f"cited page {value!r} is not a page number (pages start at 1)")
    return number


def valid_answer(answer):
    scalar = lambda v: isinstance(v, (str, int, float)) and not isinstance(v, bool)
    return scalar(answer) or (isinstance(answer, list) and all(scalar(item) for item in answer))


def parse_b1_reply(raw):
    """(answer, cited pages, evidence, error) under contract b1; any breach makes the whole reply unparseable."""
    try:
        payload = json.loads(raw)
        if not isinstance(payload, dict) or set(payload) != {"answer", "cited_pages", "evidence"}:
            raise TypeError("B1 output must contain exactly answer, cited_pages and evidence")
        answer = payload["answer"]
        if not valid_answer(answer):
            raise TypeError("answer is not a string, number, or list of strings/numbers")
        if not isinstance(payload["cited_pages"], list) or not isinstance(payload["evidence"], list):
            raise TypeError("cited_pages and evidence must be lists")
        cited = [page_number(page) for page in payload["cited_pages"]]
        evidence = []
        for item in payload["evidence"]:
            if not isinstance(item, dict) or set(item) != {"page", "quote"} or not isinstance(item["quote"], str):
                raise TypeError("each evidence item must contain exactly page and a string quote")
            evidence.append({"page": page_number(item["page"]), "quote": item["quote"]})
        if isinstance(answer, str) and answer.strip().lower() == "not answerable":
            answer = NOT_ANSWERABLE
        if answer == NOT_ANSWERABLE and (cited or evidence):
            raise ValueError("Not answerable must have empty cited_pages and evidence")
    except (json.JSONDecodeError, KeyError, TypeError, ValueError) as error:
        return "", [], [], f"unparseable model output: {error}"
    return answer, cited, evidence, None


for record in e8_log:
    if record["arm"] in ("L-B1", "L-B1r"):
        answer, cited, evidence, error = parse_b1_reply(record["raw_response"])
        assert error == record["parse_error"] and record["parsed"] == {"answer": answer, "cited_pages": cited, "evidence": evidence}
assert parse_b1_reply('{"answer": "x", "cited_pages": [2.7], "evidence": []}')[3] is not None
assert parse_b1_reply('{"answer": "not answerable", "cited_pages": [3], "evidence": []}')[3] is not None
print("parse_b1_reply reproduces all 236 frozen E8 B1 parses; it rejects fractional pages and cited abstentions.")


def utc_now():
    return datetime.now(timezone.utc).isoformat(timespec="milliseconds")


def observed_cost(record):
    if record.get("completion_tokens") is None:
        return 0.0
    prompt, cached = record.get("prompt_tokens") or 0, record.get("cached_prompt_tokens") or 0
    return max(0, prompt - cached) * PRICE["input"] + cached * PRICE["cached_input"] + record["completion_tokens"] * PRICE["output"]


def budget_charge(record):
    """A failed call can still be billed: without returned usage, charge its precomputed upper bound."""
    return observed_cost(record) if record.get("completion_tokens") is not None else record.get("cost_upper_usd", 0.0)


def load_log(path, plan):
    records = read_jsonl(path) if Path(path).is_file() else []
    by_index = {item["execution_index"]: item for item in plan}
    successful, attempts = {}, Counter()
    for line, record in enumerate(records, start=1):
        index = record.get("execution_index")
        assert index in by_index and record.get("fingerprint") == by_index[index]["fingerprint"], f"log line {line} is not in the plan"
        attempts[index] += 1
        assert attempts[index] <= 2, f"execution index {index} has more than two attempts"
        if record.get("status") == "ok":
            assert index not in successful, f"two successful replies for execution index {index}"
            successful[index] = record
    return records, successful, attempts


def execute(create, run_dir, plan, *, budget_usd, prior_spend=0.0, stop_after=None):
    """Execute or resume a plan by execution index. Never retries silently, never serves a repeat from a cache."""
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    log_path = run_dir / "execution_log.jsonl"
    records, successful, attempts = load_log(log_path, plan)
    spent = prior_spend + sum(budget_charge(r) for r in records)
    observed = sum(observed_cost(r) for r in records)
    models = {r.get("served_model") for r in successful.values()}
    fingerprints = {r.get("system_fingerprint") for r in successful.values()}
    done_now = 0
    for item in plan:
        index = item["execution_index"]
        if index in successful:
            continue
        if attempts[index] >= 2:
            raise RuntimeError(f"execution index {index} exhausted its two attempts")
        request_arm = "B1s" if item["arm"] == "B1s" else "F8"
        request = rebuild(item["qid"], request_arm)
        upper = upper_cost(loggable(request))
        if spent + upper > budget_usd:
            raise ValueError(f"the next request could exceed the cumulative E9 ceiling: ${spent + upper:.6f} > ${budget_usd:.6f}")
        record = {**item, "attempt": attempts[index] + 1, "started_at": utc_now(), "status": "started",
                  "cost_upper_usd": upper, "request": loggable(request)}
        started = time.perf_counter()
        try:
            response = create(**request)
        except Exception as error:
            record.update(status="api_error", error=f"{type(error).__name__}: {str(error)[:300]}", completed_at=utc_now(),
                          latency_s=round(time.perf_counter() - started, 3))
            with log_path.open("a") as handle:
                handle.write(json.dumps(record) + "\n")
            raise RuntimeError(f"E9 stopped after an API failure at execution index {index}; resume is explicit") from error
        choice = response.choices[0]
        raw = choice.message.content or ""
        answer, cited, evidence, error = parse_b1_reply(raw)
        if choice.finish_reason != "stop" and error is None:
            error = f"reply did not finish normally (finish_reason={choice.finish_reason!r})"
        usage = getattr(response, "usage", None)
        details = getattr(usage, "prompt_tokens_details", None) if usage else None
        record.update(status="ok" if error is None else "contract_error", completed_at=utc_now(),
                      latency_s=round(time.perf_counter() - started, 3), served_model=getattr(response, "model", None),
                      system_fingerprint=getattr(response, "system_fingerprint", None), finish_reason=choice.finish_reason,
                      raw_response=raw, parsed={"answer": answer, "cited_pages": cited, "evidence": evidence}, parse_error=error,
                      prompt_tokens=getattr(usage, "prompt_tokens", None), completion_tokens=getattr(usage, "completion_tokens", None),
                      cached_prompt_tokens=getattr(details, "cached_tokens", 0) if details else 0)
        record["list_price_cost_usd"] = observed_cost(record)
        new_models, new_fingerprints = models | {record["served_model"]}, fingerprints | {record["system_fingerprint"]}
        if len(new_models) != 1 or len(new_fingerprints) != 1:
            record["status"] = "identity_mismatch"
        with log_path.open("a") as handle:
            handle.write(json.dumps(record) + "\n")
        records.append(record)
        attempts[index] += 1
        spent += budget_charge(record)
        observed += record["list_price_cost_usd"]
        if record["status"] == "identity_mismatch":
            raise RuntimeError(f"served model or system fingerprint changed: {sorted(map(str, new_models))}, {sorted(map(str, new_fingerprints))}")
        if error is not None:
            raise RuntimeError(f"E9 stopped on a contract failure at execution index {index}: {error}")
        models, fingerprints = new_models, new_fingerprints
        successful[index] = record
        done_now += 1
        if stop_after is not None and done_now >= stop_after:
            break
    summary = {"complete": len(successful) == len(plan), "successful": len(successful), "planned": len(plan),
               "attempts": len(records), "observed_list_price_cost_usd": round(observed, 9),
               "budget_accounted_usd_including_prior": round(spent, 9), "served_models": sorted(map(str, models)),
               "system_fingerprints": sorted(map(str, fingerprints)), "updated_at": utc_now()}
    (run_dir / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
    return summary


class MockCreate:
    """Interface test double: returns a valid abstention with exact usage. Never a project result."""
    def __init__(self, *, fail_on=None, malformed_on=None, model_change_on=None, fingerprint_change_on=None):
        self.calls, self.fail_on, self.malformed_on = 0, fail_on, malformed_on
        self.model_change_on, self.fingerprint_change_on = model_change_on, fingerprint_change_on

    def __call__(self, **request):
        self.calls += 1
        if self.calls == self.fail_on:
            raise TimeoutError("mock transport interruption")
        raw = "{}" if self.calls == self.malformed_on else json.dumps({"answer": NOT_ANSWERABLE, "cited_pages": [], "evidence": []})
        usage = SimpleNamespace(prompt_tokens=prompt_tokens(loggable(request)), completion_tokens=12,
                                prompt_tokens_details=SimpleNamespace(cached_tokens=0))
        return SimpleNamespace(choices=[SimpleNamespace(message=SimpleNamespace(content=raw), finish_reason="stop")],
                               model=MODEL + ("-changed" if self.calls == self.model_change_on else ""),
                               system_fingerprint="fp_changed" if self.calls == self.fingerprint_change_on else None, usage=usage)


mock_plan = FULL_PLAN[:10]
with tempfile.TemporaryDirectory() as tmp:
    mock = MockCreate()
    partial = execute(mock, Path(tmp) / "a", mock_plan, budget_usd=BUDGET_USD, stop_after=3)
    assert not partial["complete"] and partial["successful"] == 3
    complete = execute(mock, Path(tmp) / "a", mock_plan, budget_usd=BUDGET_USD)
    assert complete["complete"] and mock.calls == 10, complete
    failing = MockCreate(fail_on=2)
    try:
        execute(failing, Path(tmp) / "b", mock_plan, budget_usd=BUDGET_USD)
        raise AssertionError("an API failure must stop the run")
    except RuntimeError as err:
        assert "API failure" in str(err)
    resumed = execute(MockCreate(), Path(tmp) / "b", mock_plan, budget_usd=BUDGET_USD)
    assert resumed["complete"] and resumed["attempts"] == 11
    assert resumed["budget_accounted_usd_including_prior"] > resumed["observed_list_price_cost_usd"]  # the failure is charged its bound
    refused = MockCreate()
    try:
        execute(refused, Path(tmp) / "c", mock_plan, budget_usd=1e-9)
        raise AssertionError("the budget must refuse")
    except ValueError as err:
        assert "ceiling" in str(err) and refused.calls == 0
    try:
        execute(MockCreate(), Path(tmp) / "c2", mock_plan, budget_usd=BUDGET_USD, prior_spend=BUDGET_USD)
        raise AssertionError("prior spend must count")
    except ValueError:
        pass
    for kwargs, message in (({"malformed_on": 1}, "contract failure"), ({"model_change_on": 2}, "changed"),
                            ({"fingerprint_change_on": 2}, "changed")):
        try:
            execute(MockCreate(**kwargs), Path(tmp) / f"d{len(message)}{list(kwargs)[0]}", mock_plan, budget_usd=BUDGET_USD)
            raise AssertionError(f"{kwargs} must stop the run")
        except RuntimeError as err:
            assert message in str(err), err
print("Executor interface tests passed with a mock: resume, API failure (charged at its bound), cumulative and prior-spend budget,"
      " malformed output, served-model and system-fingerprint change.")

parse_b1_reply reproduces all 236 frozen E8 B1 parses; it rejects fractional pages and cited abstentions.


Executor interface tests passed with a mock: resume, API failure (charged at its bound), cumulative and prior-spend budget, malformed output, served-model and system-fingerprint change.


In [15]:
# The paid gate. offline: nothing is sent. smoke: the 6 F8 smoke calls. full: the 354 planned calls (after a passed smoke).
E9_SMOKE_RUNS, E9_FULL_RUNS = PROJECT_ROOT / "experiments" / "runs" / "e9_smoke", PROJECT_ROOT / "experiments" / "runs" / "e9"


def committed(paths):
    return subprocess.run(["git", "diff", "--quiet", "HEAD", "--", *paths], cwd=PROJECT_ROOT).returncode == 0 and \
        not subprocess.run(["git", "ls-files", "--others", "--exclude-standard", "--", *paths], cwd=PROJECT_ROOT,
                           capture_output=True, text=True).stdout.strip()


def smoke_gate(run_dir):
    records = read_jsonl(run_dir / "execution_log.jsonl")
    ok = [r for r in records if r["status"] == "ok"]
    gate = {
        "6/6 replies": len(ok) == 6,
        "finish_reason stop": all(r["finish_reason"] == "stop" for r in ok),
        "strict schema parsed": all(r["parse_error"] is None for r in ok),
        "one served model": len({r["served_model"] for r in ok}) == 1,
        "served model is the requested alias": {r["served_model"] for r in ok} == {MODEL},
        "billed prompt tokens equal the token model": all(r["prompt_tokens"] == prompt_tokens(r["request"]) for r in ok),
        "smoke cost within $0.02": sum(observed_cost(r) for r in records) <= SMOKE_BUDGET_USD,
    }
    return gate, sum(budget_charge(r) for r in records)


def latest(runs_dir, suffix):
    runs = sorted(p for p in runs_dir.glob(f"*_{suffix}") if (p / "summary.json").is_file())
    return runs[-1] if runs else None


if MODE == "offline":
    print("Paid gate closed (VISUALDOC_E9_MODE=offline): no client was created and no request was sent.")
else:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / ".env")
    smoke_dir = latest(E9_SMOKE_RUNS, "smoke")
    smoke_ok, smoke_spend = (smoke_gate(smoke_dir) if smoke_dir else ({}, 0.0))
    readiness = {
        "Stage 0 passed or fallback fired": STAGE0_PASSED or FALLBACK_RUNS,
        "OPENAI_API_KEY present (value never printed)": bool(os.getenv("OPENAI_API_KEY")),
        "technical addendum frozen and equal to this design": ADDENDUM_FROZEN,
        "notebook, spec and addendum committed": committed(["notebooks/08_adaptive_evidence.ipynb",
                                                           "experiments/specs/e9_adaptive_evidence_v1.json",
                                                           "experiments/specs/e9_adaptive_evidence_v1_addendum.json"]),
    }
    if MODE == "full":
        readiness["smoke run passed every gate"] = bool(smoke_ok) and all(smoke_ok.values())
    display(pd.Series(readiness, name="ready").to_frame())
    assert all(readiness.values()), "E9 paid gate closed: no request was sent"
    from openai import OpenAI
    client = OpenAI(max_retries=0, timeout=120)
    runs_dir, plan, suffix = (E9_SMOKE_RUNS, SMOKE_PLAN, "smoke") if MODE == "smoke" else (E9_FULL_RUNS, FULL_PLAN, "full")
    requested = os.getenv("VISUALDOC_E9_RUN_ID", "").strip()
    if requested:
        run_dir = runs_dir / requested
        assert run_dir.is_dir(), f"resume run does not exist: {run_dir}"
    else:
        run_id = f"{datetime.now(timezone.utc):%Y%m%dT%H%M%S%fZ}_{uuid.uuid4().hex[:8]}_{suffix}"
        run_dir = runs_dir / run_id
        run_dir.mkdir(parents=True, exist_ok=False)
        git_commit = subprocess.run(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, capture_output=True, text=True, check=True).stdout.strip()
        (run_dir / "settings.json").write_text(json.dumps({"run_id": run_id, "mode": MODE, "git_commit": git_commit,
            "spec_sha256": SPEC_SHA256, "addendum_sha256": sha256_bytes(ADDENDUM_PATH.read_bytes()),
            "budget_usd": BUDGET_USD, "prior_spend_usd": smoke_spend if MODE == "full" else 0.0, "plan": plan}, indent=2) + "\n")
    settings = json.loads((run_dir / "settings.json").read_text())
    budget = SMOKE_BUDGET_USD if MODE == "smoke" else BUDGET_USD
    run_summary = execute(client.chat.completions.create, run_dir, plan, budget_usd=budget, prior_spend=settings["prior_spend_usd"])
    display(pd.Series(run_summary, name=f"E9 {MODE}").to_frame())
    if MODE == "smoke":
        gate, _ = smoke_gate(run_dir)
        (run_dir / "smoke_gate.json").write_text(json.dumps(gate, indent=2) + "\n")
        display(pd.Series(gate, name="smoke gate").to_frame())
    print(f"E9 {MODE} outputs: {run_dir.relative_to(PROJECT_ROOT)}")

Paid gate closed (VISUALDOC_E9_MODE=offline): no client was created and no request was sent.


## Stage 2 evaluation and the frozen decision rules

**Scoring.** S-v1 (notebook 03 cell `f653d232`; correct = score > 0) is primary and decides the verdict. S-v2 is reported beside it. A gray reply reuses an owner judgment only when its blind identity (document, evidence pages, question, format, reference, reply) hashes to an item already labelled in `experiments/frozen/scoring_v2` or `experiments/frozen/e8_scoring_v2`. Every other gray reply goes to a new arm-blind queue for the **owner**: no model judgment is substituted.

The two scorer cells below are **verbatim copies** of notebook 07's `b1-scorer-v1` and `b1-scorer-v2`. The first re-checks itself against notebook 03's frozen cell, and the check cell pins both copies by SHA-256. A copy, rather than an `exec` of the other notebook, keeps the scoring code visible here and lets its `inspect`-based self-check run.

**Statistics** (spec `stage2`).
- Unit: answerable question (102), clustered by document (14).
- Δ = replicate-averaged accuracy of the policy minus B1s's.
- Document-cluster bootstrap: 10,000 draws, seed 20261128, one-sided 5th percentile.
- Exact document sign-flip test, enumerating all $2^{14}$ sign patterns of the per-document summed differences.
- Fixed / broken counts per replicate.

In [16]:
# Names the verbatim scorer cells expect (as in notebook 07's setup), then the dev questions of the 14 held-out documents only.
import ast
import inspect
import re
from math import isclose

NB03_PATH = PROJECT_ROOT / "notebooks" / "03_evaluation.ipynb"
assert NB03_PATH.is_file(), "notebook 03 is tracked in git: restore it"
QUESTIONS_PATH = PROJECT_ROOT / "data" / "processed" / "mmlongbench_doc" / "questions.parquet"
assert QUESTIONS_PATH.is_file(), "run 00_document_processing to build questions.parquet"
HELDOUT14 = sorted(q["doc_id"].unique())
_all_questions = pd.read_parquet(QUESTIONS_PATH)
dev = _all_questions[_all_questions["doc_id"].isin(HELDOUT14)].set_index("qid").copy()
del _all_questions  # the test pool is dropped here, before any row is looked at
dev["evidence_pages"] = dev["evidence_pages"].map(lambda pages: [int(p) for p in pages])
assert set(dev.index) == set(q.index) and len(dev) == 118
print(f"Development questions loaded for the 14 held-out documents only: {len(dev)} (test-pool rows dropped unread).")

Development questions loaded for the 14 held-out documents only: 118 (test-pool rows dropped unread).


In [17]:
# S-v1: the frozen MMLongBench-Doc scorer (notebook 03, cell f653d232), written out unchanged. Correct = score > 0; strict = score == 1.
def levenshtein_distance(s1, s2):
    if len(s1) > len(s2):
        s1, s2 = s2, s1
    distances = range(len(s1) + 1)
    for i2, c2 in enumerate(s2):
        new_distances = [i2 + 1]
        for i1, c1 in enumerate(s1):
            new_distances.append(distances[i1] if c1 == c2 else 1 + min(distances[i1], distances[i1 + 1], new_distances[-1]))
        distances = new_distances
    return distances[-1]


def anls_compute(groundtruth, prediction, threshold=0.5):
    length = max(len(groundtruth.upper()), len(prediction.upper()))
    anls = 1.0 - (0.0 if length == 0 else levenshtein_distance(groundtruth, prediction) / length)
    return 0.0 if anls <= threshold else anls


def is_float_equal(reference, prediction, include_percentage=False, is_close=False):
    def get_precision(value):
        text = str(value)
        return len(text.split(".")[-1]) if "." in text else 3

    reference = float(str(reference).strip().rstrip("%").strip())
    try:
        prediction = float(str(prediction).strip().rstrip("%").strip())
    except (TypeError, ValueError):
        return False
    for item in ([reference / 100, reference, reference * 100] if include_percentage else [reference]):
        if is_close and isclose(item, prediction, rel_tol=0.01):
            return True
        precision = max(min(get_precision(prediction), get_precision(item)), 2)
        if round(prediction, precision) == round(item, precision):
            return True
    return False


def get_clean_string(s):
    s = str(s).lower().strip()
    # Official quirk kept: the code calls s.rstrip("mile") etc. without assigning the result, so unit suffixes stay.
    s = re.sub(r"\s*\([^)]*\)", "", s).strip()  # remove parenthesised text
    s = re.sub(r"^['\"]|['\"]$", "", s).strip()  # remove surrounding quotes
    s = s.strip().lstrip("$").strip()
    return s.strip().rstrip("%").strip()


EXACT_MATCH_PATTERNS = [
    r"\b\d+(-\d+|\s\d+)?\b",  # phone-like numbers
    r"\b\d{4}[-\s]\d{2}[-\s]\d{2}\b",  # YYYY-MM-DD
    r"\b\d{4}[-\s]\d{2}\b",  # YYYY-MM
    r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}",  # e-mail address
]


def is_exact_match(s):
    return ("https://" in s or s.endswith(".py") or s.endswith("ipynb") or s.startswith("page") or "a.m." in s or "p.m." in s
            or any(re.fullmatch(pattern, s) for pattern in EXACT_MATCH_PATTERNS))


def isfloat(value):
    try:
        float(value)
        return True
    except (TypeError, ValueError):
        return False


def as_list(value):
    if isinstance(value, str) and value.startswith("["):
        try:
            value = ast.literal_eval(value)
        except (ValueError, SyntaxError):
            pass  # deviation: the official eval() would raise here
    return value if isinstance(value, list) else [value]


def eval_score(gt, pred, answer_type):
    if answer_type == "Int":
        try:
            gt, pred = int(gt), int(float(pred))
        except Exception:  # the official code uses a bare except
            pred = ""
        score = gt == pred
    elif answer_type == "Float":
        try:
            gt, pred = float(get_clean_string(str(gt))), float(get_clean_string(str(pred)))
        except Exception:
            pred = ""
        score = is_float_equal(gt, pred, include_percentage=True, is_close=True)
    elif answer_type in ("Str", "None"):
        gt, pred = get_clean_string(gt), get_clean_string(pred)
        score = gt == pred if is_exact_match(gt) else anls_compute(gt, pred)
    else:  # List
        gt, pred = as_list(gt), as_list(pred)
        if len(gt) != len(pred):
            score = 0.0
        else:
            gt, pred = sorted(get_clean_string(i) for i in gt), sorted(get_clean_string(i) for i in pred)
            score = "-".join(gt) == "-".join(pred) if isfloat(gt[0]) or is_exact_match(gt[0]) else min(anls_compute(g, p) for g, p in zip(gt, pred))
    return float(score)


# 1. Unchanged: every function above has the same source as the frozen cell, and so does the pattern list.
frozen_cell = next("".join(c["source"]) for c in json.loads(NB03_PATH.read_text())["cells"] if c.get("id") == "f653d232")
frozen_tree = ast.parse(frozen_cell)
frozen_functions = {node.name: ast.get_source_segment(frozen_cell, node) for node in frozen_tree.body if isinstance(node, ast.FunctionDef)}
SCORER_FUNCTIONS = ["levenshtein_distance", "anls_compute", "is_float_equal", "get_clean_string", "is_exact_match", "isfloat", "as_list", "eval_score"]
assert set(SCORER_FUNCTIONS) <= set(frozen_functions)
assert all(inspect.getsource(globals()[name]).strip() == frozen_functions[name].strip() for name in SCORER_FUNCTIONS)
frozen_patterns = next(ast.literal_eval(node.value) for node in frozen_tree.body
                       if isinstance(node, ast.Assign) and getattr(node.targets[0], "id", None) == "EXACT_MATCH_PATTERNS")
assert frozen_patterns == EXACT_MATCH_PATTERNS

# 2. The 17 hand-traced cases of notebook 03.
SCORING_CASES = [  # reference, prediction, format, expected
    ("Less well-off", "less well off", "Str", 1 - 1 / 13), ("Pew Research Center", "Pew Research Centre", "Str", 1 - 2 / 19),
    ("Latinos", "Asian Americans", "Str", 0.0), ("Not answerable", "Not answerable", "None", 1.0), ("Not answerable", "The report does not say", "None", 0.0),
    ("12", "12.0", "Int", 1.0), ("12", "twelve", "Int", 0.0), ("18.29%", "18.3", "Float", 1.0), ("0.1", "10%", "Float", 1.0), ("51%", "49", "Float", 0.0),
    ("['Circle', 'Rectangle']", "['rectangle', 'circle']", "List", 1.0), ("['White', '10%']", "['White']", "List", 0.0),
    ("['2014', '2016']", "['2016', '2014']", "List", 1.0), ("https://www.example.com", "www.example.com", "Str", 0.0), ("2016-06", "2016-06", "Str", 1.0),
    ("$1,200 (approx.)", "1,200", "Str", 1.0), ("5 miles", "5", "Str", 0.0)]
assert len(SCORING_CASES) == 17 and all(isclose(eval_score(r, p, f), e, abs_tol=1e-9) for r, p, f, e in SCORING_CASES)
print(f"S-v1: {len(SCORER_FUNCTIONS)} functions and the pattern list equal notebook 03's frozen cell f653d232; {len(SCORING_CASES)} hand-traced cases pass.")

S-v1: 8 functions and the pattern list equal notebook 03's frozen cell f653d232; 17 hand-traced cases pass.


In [18]:
# S-v2: a deterministic layer over S-v1 that returns (class, reason) with class in {correct, wrong, gray}. Gray goes to blind adjudication.
SCALE_WORDS = {"thousand": 1e3, "k": 1e3, "million": 1e6, "millions": 1e6, "mn": 1e6, "billion": 1e9, "billions": 1e9, "bn": 1e9, "trillion": 1e12}
NUMBER_PATTERN = re.compile(r"(\()?(-?\d[\d,]*(?:\.\d+)?)(\))?\s*(%|(?:percent|thousand|millions?|billions?|trillion|bn|mn|k)\b)?")
NUMERIC_TEXT_RESIDUE = re.compile(r"us\$|usd|eur|gbp|percent|thousand|millions?|billions?|trillion|\bbn\b|\bmn\b|\bk\b|[$€£%(),.\-\s\d]")
POWERS_OF_1000 = (1e-9, 1e-6, 1e-3, 1e3, 1e6, 1e9)  # "m" and "b" are not scale words: metres, bytes


def numbers_in(value):
    """Every number in a reply or reference, after removing currency signs; a scale word multiplies, '(x)' is negative."""
    if isinstance(value, bool):
        return []
    if isinstance(value, (int, float)):
        return [float(value)]
    text = re.sub(r"us\$|usd|eur|gbp|[$€£]", " ", str(value).lower())
    found = []
    for open_paren, digits, close_paren, unit in NUMBER_PATTERN.findall(text):
        number = float(digits.replace(",", ""))
        found.append((-number if open_paren and close_paren else number) * SCALE_WORDS.get(unit, 1.0))
    return found


def numeric_text(value):
    """True when a value is one number with at most currency, %, separators and a scale word around it."""
    if isinstance(value, bool):
        return False
    if isinstance(value, (int, float)):
        return True
    return NUMERIC_TEXT_RESIDUE.sub("", str(value).lower()) == "" and len(numbers_in(value)) == 1


def same_value(reference, prediction, answer_format):
    # Float: within 1 %, including the official x100 / /100 percentage variants; Int: exactly equal.
    if answer_format == "Float":
        return any(isclose(candidate, prediction, rel_tol=0.01, abs_tol=1e-9) for candidate in (reference, reference / 100, reference * 100))
    return isclose(reference, prediction, rel_tol=0.0, abs_tol=1e-9)


def numeric_class(reference, prediction, answer_format):
    """correct / scale / among / wrong / None (the reference is not one number)."""
    ref, pred = numbers_in(reference), numbers_in(prediction)
    if len(ref) != 1 or not pred:
        return None if len(ref) != 1 else "wrong"
    if len(pred) > 1:
        return "among" if any(same_value(ref[0], p, answer_format) for p in pred) else "wrong"
    if same_value(ref[0], pred[0], answer_format):
        return "correct"
    if any(same_value(ref[0] * f, pred[0], answer_format) for f in POWERS_OF_1000):
        return "scale"
    return "wrong"


def normalise(value):
    text = json.dumps(value) if isinstance(value, list) else str(value)
    return " ".join(re.sub(r"[^a-z0-9%.]+", " ", text.lower()).split())


def contains_phrase(haystack, needle):
    return bool(needle) and f" {needle} " in f" {haystack} "


def canonical_item(item):
    """A list item for comparison: a number as a number ('$35,000' = 35000), anything else normalised text."""
    return f"{numbers_in(item)[0]:.6g}" if numeric_text(item) else normalise(item)


def item_set(value):
    return {canonical_item(item) for item in as_list(value)} - {""}


def items_match(a, b):
    """Equal, one a phrase of the other, or ANLS above 0.5 (S-v1's threshold)."""
    return a == b or contains_phrase(a, b) or contains_phrase(b, a) or anls_compute(a, b) > 0


# References the frozen E6 error audit labelled as annotation issues: scored by S-v1 as usual, sent to adjudication by S-v2.
e6_audit = pd.read_csv(FROZEN_DIR / "e7" / "design" / "e6_error_audit.csv")
DEBATABLE_QIDS = set(e6_audit.loc[e6_audit["category"] == "annotation issue", "qid"])


NUMERIC_REASON = {"correct": ("correct", "numeric match after normalisation (currency signs, separators, %, scale words)"),
                  "scale": ("gray", "same number up to a factor of 1,000^n (unit scale ambiguous)"),
                  "among": ("gray", "the reference value is one of several numbers in the reply"),
                  "wrong": ("wrong", "numeric mismatch")}


def score_v2(reference, prediction, answer_format, debatable=False):
    s1 = eval_score(reference, prediction, answer_format)
    is_abstention = isinstance(prediction, str) and prediction.strip().lower() == NOT_ANSWERABLE.lower()
    if answer_format == "None":
        return ("correct", "exact abstention") if s1 >= 1 else ("wrong", "answered an unanswerable question")
    if is_abstention:
        return "wrong", "false abstention"
    if s1 >= 1:
        return "correct", "S-v1 full match"
    if debatable:
        return "gray", "reference flagged as debatable by the frozen E6 audit"
    if answer_format in ("Int", "Float"):
        return NUMERIC_REASON[numeric_class(reference, prediction, answer_format) or "wrong"]
    if answer_format == "Str":
        if numeric_text(reference) and numeric_text(prediction):
            numeric = numeric_class(reference, prediction, "Float")
            if numeric in ("correct", "scale"):
                return NUMERIC_REASON[numeric]
        ref, pred = normalise(reference), normalise(prediction)
        if ref and ref == pred:
            return "correct", "equal after case and punctuation normalisation"
        if 0 < s1 < 1:
            return "gray", "partial ANLS credit (0.5 < ANLS < 1)"
        if contains_phrase(pred, ref):
            return "gray", "the reference is a phrase inside a longer reply"
        if len(pred.split()) >= 2 and contains_phrase(ref, pred):
            return "gray", "the reply is a phrase inside the reference"
        return "wrong", "string mismatch"
    # List: a reply missing a reference item is wrong (the benchmark asks for the whole list); only full matches go to a human.
    ref_items, pred_items = item_set(reference), item_set(prediction)
    if ref_items and ref_items == pred_items:
        return "correct", "same items after normalisation"
    if ref_items and all(any(items_match(r, p) for p in pred_items) for r in ref_items):
        if len(pred_items) > len(ref_items):
            return "gray", "every reference item present, with extra items"
        return "gray", "every reference item matched up to formatting"
    return "wrong", "a reference item is missing"


# Hand checks on invented values, including what S-v2 cannot settle (negation, debatable references): those stay gray or wrong for a human.
V2_CASES = [  # reference, reply, format, expected class
    ("Not answerable", NOT_ANSWERABLE, "None", "correct"), ("Not answerable", "7", "None", "wrong"), ("7", NOT_ANSWERABLE, "Int", "wrong"),
    ("1234", "1,234 units", "Int", "correct"), ("12", "12%", "Int", "correct"), ("15", "between 15 and 20", "Int", "gray"),
    ("3.5", "3.5 million", "Float", "gray"), ("1200", "$1.2 billion", "Float", "gray"), ("42.0", "17.3", "Float", "wrong"),
    ("300 million", "300", "Str", "gray"), ("Northern region", "The northern region had the most visitors", "Str", "gray"),
    ("Special Agent Ana Lee", "Ana Lee", "Str", "gray"), ("Ana Lee", "Bo Chen", "Str", "wrong"),
    ("Paris", "Not Paris but Lyon", "Str", "gray"),  # failure case: containment cannot see the negation; the adjudicator must
    ("Less well-off", "less well off", "Str", "correct"), ("Pew Research Center", "Pew Research Centre", "Str", "gray"),
    ("['Circle', 'Rectangle']", ["rectangle", "circle"], "List", "correct"), ("['$35,000', '$40,684']", [35000, 40684], "List", "correct"),
    ("['Apple Inc', 'Banana']", ["apple", "banana co"], "List", "gray"), ("['a', 'b']", ["a", "b", "c"], "List", "gray"),
    ("['a', 'b', 'c']", ["a", "b"], "List", "wrong"), ("['a']", ["x"], "List", "wrong")]
assert score_v2("12", "13", "Int", debatable=True) == ("gray", "reference flagged as debatable by the frozen E6 audit")
assert score_v2("12", "12", "Int", debatable=True)[0] == "correct"  # S-v1 = 1 stays correct
observed = {(r, str(p), f): score_v2(r, p, f)[0] for r, p, f, _ in V2_CASES}
assert all(observed[(r, str(p), f)] == expected for r, p, f, expected in V2_CASES), {k: v for k, v in observed.items()}
print(f"S-v2: {len(V2_CASES)} invented cases classified as expected (including a negation that only a human can resolve).")

S-v2: 22 invented cases classified as expected (including a negation that only a human can resolve).


In [19]:
# The two scorer cells above are byte-identical to notebook 07's (pinned SHA-256), and therefore to the scoring every frozen verdict used.
SCORER_PINS = {"b1-scorer-v1": "329f91f3ceda29146ec8a0c2467f11fdf7fdbd66c1e6af29123e1983a3fc7ee7",
               "b1-scorer-v2": "902708ba537d580db8c67858fe9a2231e5b546f1538a2b955a3eadff551b8707"}
this_nb = {c.get("id"): "".join(c["source"]) for c in json.loads((PROJECT_ROOT / "notebooks" / "08_adaptive_evidence.ipynb").read_text())["cells"]}
nb07_cells = {c.get("id"): "".join(c["source"]) for c in json.loads((PROJECT_ROOT / "notebooks" / "07_baseline_consolidation.ipynb").read_text())["cells"]}
for nb07_id, here_id in (("b1-scorer-v1", "e9-scorer-v1"), ("b1-scorer-v2", "e9-scorer-v2")):
    assert sha256_bytes(this_nb[here_id].encode()) == SCORER_PINS[nb07_id] == sha256_bytes(nb07_cells[nb07_id].encode()), here_id
print("Scorer cells equal notebook 07's pinned copies; S-v1 equals notebook 03's frozen cell f653d232.")

Scorer cells equal notebook 07's pinned copies; S-v1 equals notebook 03's frozen cell f653d232.


In [20]:
# E9 evaluation. It reads a completed full run (the frozen package once it exists), never calls an API, and applies the frozen rules.
E9_FROZEN = FROZEN_DIR / "e9"
requested_eval = os.getenv("VISUALDOC_E9_EVAL_RUN", "").strip()
frozen_runs = sorted(E9_FROZEN.glob("run_*_full")) if E9_FROZEN.is_dir() else []
live_runs = sorted(p.parent for p in E9_FULL_RUNS.glob("*_full/summary.json") if json.loads(p.read_text()).get("complete"))
EVAL_RUN = PROJECT_ROOT / requested_eval if requested_eval else (frozen_runs[-1] if frozen_runs else (live_runs[-1] if live_runs else None))
E9_EVALUATED = EVAL_RUN is not None
if not E9_EVALUATED:
    print("No completed E9 full run exists yet: Stage 2 evaluation is waiting on the paid run (readiness gate above).")
else:
    records = read_jsonl(EVAL_RUN / "execution_log.jsonl")
    settings = json.loads((EVAL_RUN / "settings.json").read_text())
    assert settings["plan"] == FULL_PLAN, "the run's plan is not this notebook's frozen plan"
    assert settings["spec_sha256"] == SPEC_SHA256
    ok = [r for r in records if r["status"] == "ok"]
    assert len(ok) == 354 and len({r["execution_index"] for r in ok}) == 354
    assert {r["served_model"] for r in ok} == {MODEL} and len({r["system_fingerprint"] for r in ok}) == 1
    assert all(r["finish_reason"] == "stop" and r["parse_error"] is None for r in ok)

    def citation_valid(answer, cited, supplied):
        """Notebook 03's form check: every cited page was supplied, and a non-abstention cites at least one page."""
        return set(cited) <= set(supplied) and (answer == NOT_ANSWERABLE or bool(cited))

    rows = []
    for r in ok:
        qid, arm = r["qid"], r["arm"]
        reference, fmt = dev.at[qid, "answer"], dev.at[qid, "answer_format"]
        answer, cited = r["parsed"]["answer"], r["parsed"]["cited_pages"]
        supplied = REQUESTS[(qid, "B1s" if arm == "B1s" else "F8")]["pages"]
        s1 = eval_score(reference, answer, fmt)
        v2_class, v2_reason = score_v2(reference, answer, fmt, debatable=qid in DEBATABLE_QIDS)
        rows.append({"qid": qid, "doc_id": dev.at[qid, "doc_id"], "arm": arm, "answerable": bool(dev.at[qid, "answerable"]),
                     "s1_score": s1, "s1_correct": s1 > 0, "v2_class": v2_class, "v2_reason": v2_reason,
                     "schema_valid": r["parse_error"] is None, "citation_valid": citation_valid(answer, cited, supplied),
                     "abstained": answer == NOT_ANSWERABLE, "prediction": answer, "pages_read": len(supplied),
                     "prompt_tokens": r["prompt_tokens"], "completion_tokens": r["completion_tokens"],
                     "cost_usd": r["list_price_cost_usd"], "latency_s": r["latency_s"]})
    paid = pd.DataFrame(rows)
    assert paid.groupby("arm").size().to_dict() == {"B1s": 118, "F8a": 118, "F8b": 118}

    # A1 is derived, not called: the F8 reply where the frozen trigger fires, the B1s reply elsewhere.
    TRIGGERED = set(DESIGN["stage0"]["triggered_questions"])
    by_arm = {arm: frame.set_index("qid") for arm, frame in paid.groupby("arm")}
    derived = []
    for name, deep in (("A1a", "F8a"), ("A1b", "F8b")):
        for qid in QIDS:
            source = deep if qid in TRIGGERED else "B1s"
            derived.append({**by_arm[source].loc[qid].to_dict(), "qid": qid, "arm": name, "source_arm": source})
    scores = pd.concat([paid.assign(source_arm=paid["arm"]), pd.DataFrame(derived)], ignore_index=True)

    # S-v2: reuse an owner label only for an identical blind item; queue the rest for the owner (never a model judgment).
    identity_cols = ["document", "evidence_pages", "question", "answer_format", "reference", "reply"]

    def blind_row(qid, prediction):
        return {"document": dev.at[qid, "doc_id"], "evidence_pages": " ".join(map(str, dev.at[qid, "evidence_pages"])),
                "question": dev.at[qid, "question"], "answer_format": dev.at[qid, "answer_format"],
                "reference": dev.at[qid, "answer"], "reply": json.dumps(prediction) if isinstance(prediction, list) else str(prediction)}

    def blind_identity_sha256(row):
        payload = {column: str(row[column]) for column in identity_cols}
        return sha256_bytes(json.dumps(payload, sort_keys=True, ensure_ascii=False).encode())

    owner_labels = {}
    for package in ("scoring_v2", "e8_scoring_v2"):
        checks = verify_sha256sums(FROZEN_DIR / package)
        assert checks and all(checks.values()), f"experiments/frozen/{package} differs from its SHA256SUMS"
        labels = pd.read_csv(FROZEN_DIR / package / "adjudication_labels.csv", dtype=str).fillna("")
        owner_labels.update(dict(zip(labels["identity_sha256"], labels["label"])))
    E9_LABELS_DIR = FROZEN_DIR / "e9_scoring_v2"
    if (E9_LABELS_DIR / "SHA256SUMS").is_file():
        checks = verify_sha256sums(E9_LABELS_DIR)
        assert checks and all(checks.values()), "experiments/frozen/e9_scoring_v2 differs from its SHA256SUMS"
        labels = pd.read_csv(E9_LABELS_DIR / "adjudication_labels.csv", dtype=str).fillna("")
        owner_labels.update(dict(zip(labels["identity_sha256"], labels["label"])))

    scores["identity"] = [blind_identity_sha256(blind_row(qid, pred)) for qid, pred in zip(scores["qid"], scores["prediction"])]
    scores["v2_final"] = np.where(scores["v2_class"] == "gray", scores["identity"].map(owner_labels).fillna("pending"), scores["v2_class"])
    scores["label_reused"] = (scores["v2_class"] == "gray") & scores["identity"].isin(owner_labels)

    pending = scores[(scores["v2_final"] == "pending") & scores["answerable"] & scores["arm"].isin(["B1s", "F8a", "F8b"])]
    items = pending.drop_duplicates("identity")[["identity", "qid", "prediction"]].sample(frac=1.0, random_state=SEED + 9).reset_index(drop=True)
    items["item_id"] = [f"E9-ADJ-{i + 1:03d}" for i in range(len(items))]
    queue = pd.DataFrame([{"item_id": row.item_id, **blind_row(row.qid, row.prediction)} for row in items.itertuples()])
    if len(queue):
        queue["label (correct / wrong / unsure)"], queue["note"] = "", ""
        assert all(blind_identity_sha256(queue.iloc[i]) == items.at[i, "identity"] for i in range(len(items)))
    key = items.merge(pending.groupby("identity")["arm"].agg(lambda a: " | ".join(sorted(set(a)))).rename("arms"), on="identity")
    queue_path, key_path = OUT_DIR / "adjudication_queue.csv", OUT_DIR / "adjudication_key_SEALED.csv"
    if not queue_path.is_file() or pd.read_csv(queue_path, dtype=str).fillna("")["label (correct / wrong / unsure)"].eq("").all():
        queue.to_csv(queue_path, index=False)  # never overwrite a queue the owner has started labelling
    key[["item_id", "qid", "arms", "identity"]].to_csv(key_path, index=False)

    def arm_summary(frame):
        ans, unans = frame[frame["answerable"]], frame[~frame["answerable"]]
        return {"S-v1 correct / 102": int(ans["s1_correct"].sum()), "correct abstentions / 16": int(unans["abstained"].sum()),
                "citation validity": float(frame["citation_valid"].mean()), "schema validity": float(frame["schema_valid"].mean()),
                "S-v2 correct (auto + owner)": int((ans["v2_final"] == "correct").sum()), "S-v2 pending": int((ans["v2_final"] == "pending").sum()),
                "S-v2 unsure": int((ans["v2_final"] == "unsure").sum()), "pages read (mean)": float(frame["pages_read"].mean()),
                "prompt tokens (mean)": float(frame["prompt_tokens"].mean()), "cost USD (sum)": float(frame["cost_usd"].sum())}

    ARM_ORDER = ["B1s", "F8a", "F8b", "A1a", "A1b"]
    arm_table = pd.DataFrame({arm: arm_summary(scores[scores["arm"] == arm]) for arm in ARM_ORDER}).T
    display(arm_table.round(4))

    def correct(arm, column="s1_correct"):
        frame = scores[(scores["arm"] == arm) & scores["answerable"]].set_index("qid").sort_index()
        return frame[column].astype(float)

    ANS_DOCS = dev.loc[correct("B1s").index, "doc_id"]

    def contrast(treatment, control, seed=20261128):
        """treatment/control: per-question correctness (replicate-averaged where given). Document-cluster statistics."""
        diff = (treatment - control).sort_index()
        docs = ANS_DOCS.loc[diff.index].to_numpy()
        order = sorted(set(docs))
        sums = np.array([diff[docs == d].sum() for d in order])
        counts = np.array([(docs == d).sum() for d in order])
        picks = np.random.default_rng(seed).integers(0, len(order), size=(10_000, len(order)))
        boot = sums[picks].sum(axis=1) / counts[picks].sum(axis=1)
        signs = np.array([[1 if (mask >> i) & 1 else -1 for i in range(len(order))] for mask in range(2 ** len(order))])
        flipped = signs @ sums
        return {"delta": float(diff.mean()), "ci95": [float(x) for x in np.percentile(boot, [2.5, 97.5])],
                "p05_one_sided": float(np.percentile(boot, 5)), "sign_flip_p": float(np.mean(flipped >= sums.sum() - 1e-12))}

    def fixed_broken(treatment_arm, control_arm):
        t, c = correct(treatment_arm), correct(control_arm)
        return int(((t == 1) & (c == 0)).sum()), int(((t == 0) & (c == 1)).sum())

    b1s = correct("B1s")
    a1 = (correct("A1a") + correct("A1b")) / 2
    f8 = (correct("F8a") + correct("F8b")) / 2
    primary = contrast(a1, b1s) if A1_EXISTS else contrast(f8, b1s)
    policy = ("A1a", "A1b") if A1_EXISTS else ("F8a", "F8b")
    per_rep = {rep: {"delta": float((correct(rep) - b1s).mean()), "fixed": fixed_broken(rep, "B1s")[0], "broken": fixed_broken(rep, "B1s")[1]}
               for rep in policy}
    secondary = {"F8 - B1s": contrast(f8, b1s), "A1 - F8": contrast(a1, f8) if A1_EXISTS else None,
                 "F8 - B1s on triggered": contrast(f8[f8.index.isin(TRIGGERED)], b1s[b1s.index.isin(TRIGGERED)]) if A1_EXISTS else None,
                 "F8 - B1s on untriggered": contrast(f8[~f8.index.isin(TRIGGERED)], b1s[~b1s.index.isin(TRIGGERED)]) if A1_EXISTS else None}
    flips_f8 = int((correct("F8a") != correct("F8b")).sum())

    # Cross-session drift (descriptive): B1s against the frozen E8 L-B1 and L-B1r replies, scored with the same S-v1.
    e8_frozen = pd.read_csv(E8_TABLE_PATH)
    e8_correct = {arm: e8_frozen[(e8_frozen["arm"] == arm) & e8_frozen["answerable"]].set_index("qid")["s1_correct"].astype(float).sort_index()
                  for arm in ("L-B1", "L-B1r")}
    drift = {f"B1s vs frozen {arm}": {"B1s correct": int(b1s.sum()), "frozen correct": int(v.sum()), "flips": int((b1s != v).sum())}
             for arm, v in e8_correct.items()}

    # The frozen decision rules, in order (spec stage2.decision_rules_in_order).
    base = arm_table.loc["B1s"]
    guarded_arms = ["F8a", "F8b"] + (["A1a", "A1b"] if A1_EXISTS else [])
    guardrails = {arm: {"abstentions": arm_table.at[arm, "correct abstentions / 16"] >= base["correct abstentions / 16"] - 2,
                        "citations": arm_table.at[arm, "citation validity"] >= base["citation validity"] - 0.05,
                        "schema": arm_table.at[arm, "schema validity"] >= 0.99} for arm in guarded_arms}
    guardrails_pass = all(all(v.values()) for v in guardrails.values())
    delta = primary["delta"]
    engineering = delta >= 0.05 and all(v["delta"] >= 0.05 for v in per_rep.values())
    broken_rule = any(v["broken"] > v["fixed"] / 2 for v in per_rep.values())
    attribution_ok = (secondary["A1 - F8"]["delta"] > 0) if A1_EXISTS else True
    statistics_ok = primary["p05_one_sided"] > 0 and primary["sign_flip_p"] <= 0.05
    if not guardrails_pass:
        verdict = "regression"
    elif delta <= -0.05:
        verdict = "regression"
    elif not engineering:
        verdict = "no_meaningful_gain"
    elif broken_rule:
        verdict = "regression"
    elif not attribution_ok:
        verdict = "more_pages_not_triggering"
    elif statistics_ok and flips_f8 <= 10:
        verdict = "adopt_adaptive_acquisition" if A1_EXISTS else "adopt_fixed_k8"
    else:
        verdict = "promising_but_uncertain"

    decision = {"primary contrast": "A1 - B1s" if A1_EXISTS else "F8 - B1s", "delta (replicate-averaged)": round(delta, 4),
                "ci95": [round(x, 4) for x in primary["ci95"]], "bootstrap 5th percentile": round(primary["p05_one_sided"], 4),
                "sign-flip p": round(primary["sign_flip_p"], 5), "per replicate": per_rep, "guardrails pass": guardrails_pass,
                "engineering criterion (+0.05 in average and each replicate)": engineering, "broken > fixed/2": broken_rule,
                "A1 - F8 > 0": attribution_ok, "statistics hold": statistics_ok, "F8a vs F8b flips": flips_f8,
                "VERDICT (S-v1, frozen rules)": verdict}
    display(pd.Series({k: (json.dumps(v) if isinstance(v, (dict, list)) else v) for k, v in decision.items()}, name="E9 decision").to_frame())
    display(pd.DataFrame({k: v for k, v in secondary.items() if v}).T.round(4))
    display(pd.DataFrame(drift).T)

    # S-v2 (secondary): an interval while items are pending; a point estimate (unsure excluded) once the owner has labelled them.
    def v2_correct(arm, pending_as):
        frame = scores[(scores["arm"] == arm) & scores["answerable"]].set_index("qid").sort_index()
        final = frame["v2_final"].replace({"pending": pending_as})
        return (final[final != "unsure"] == "correct").astype(float)

    v2 = {}
    for bound in ("wrong", "correct"):
        b, a, f = v2_correct("B1s", bound), (v2_correct("A1a", bound) + v2_correct("A1b", bound)) / 2, (v2_correct("F8a", bound) + v2_correct("F8b", bound)) / 2
        common = b.index.intersection(a.index).intersection(f.index)
        v2[f"pending as {bound}"] = {"B1s": float(b[common].sum()), "A1 (avg)": float(a[common].sum()), "F8 (avg)": float(f[common].sum()),
                                     "A1 - B1s": float((a[common] - b[common]).mean()), "F8 - B1s": float((f[common] - b[common]).mean()), "n": len(common)}
    display(pd.DataFrame(v2).T.round(4))
    S_V2_FINAL = len(queue) == 0

    scores.drop(columns=["prediction", "identity"]).to_csv(OUT_DIR / "evaluation_per_question.csv", index=False)
    evaluation_summary = {"run": settings["run_id"], "arms": json.loads(arm_table.to_json(orient="index")), "decision": decision,
                          "secondary": secondary, "drift": drift, "s_v2": v2, "s_v2_final": S_V2_FINAL,
                          "new_blind_items": len(queue), "triggered_questions": len(TRIGGERED)}
    (OUT_DIR / "evaluation_summary.json").write_text(json.dumps(evaluation_summary, indent=2, default=str) + "\n")
    if E9_FROZEN in EVAL_RUN.parents:  # regression guard: the frozen package must reproduce its frozen evaluation exactly
        frozen_summary = json.loads((E9_FROZEN / "tables" / "evaluation_summary.json").read_text())
        assert json.loads(json.dumps(evaluation_summary, default=str)) == frozen_summary, "E9 no longer reproduces its frozen evaluation"
        assert {arm: int(arm_table.at[arm, "S-v1 correct / 102"]) for arm in ARM_ORDER} == {"B1s": 44, "F8a": 46, "F8b": 46, "A1a": 47, "A1b": 46}
        assert verdict == "no_meaningful_gain" and per_rep["A1a"]["fixed"] == 3 and per_rep["A1b"]["fixed"] == 2
        print("Frozen E9 evaluation reproduced exactly (B1s 44, F8a 46, F8b 46, A1a 47, A1b 46; no_meaningful_gain).")
    print(f"E9 evaluated from {EVAL_RUN.relative_to(PROJECT_ROOT)}. S-v1 verdict: {verdict}. "
          f"New blind S-v2 items for the owner: {len(queue)} ({queue_path.relative_to(PROJECT_ROOT)}).")

,S-v1 correct / 102,correct abstentions / 16,citation validity,schema validity,S-v2 correct (auto + owner),S-v2 pending,S-v2 unsure,pages read (mean),prompt tokens (mean),cost USD (sum)
B1s,44.0,9.0,1.0000,1.0,44.0,7.0,1.0,4.0000,3977.1525,0.0498
F8a,46.0,9.0,0.9915,1.0,47.0,1.0,1.0,8.0000,7641.3220,0.0734
F8b,46.0,8.0,0.9915,1.0,46.0,5.0,1.0,8.0000,7641.3220,0.0239
A1a,47.0,9.0,0.9915,1.0,47.0,6.0,1.0,5.5932,5421.2881,0.0590
A1b,46.0,9.0,0.9915,1.0,46.0,7.0,1.0,5.5932,5421.2881,0.0391


,E9 decision
primary contrast,A1 - B1s
delta (replicate-averaged),0.0245
ci95,"[0.0, 0.0505]"
bootstrap 5th percentile,0.0052
sign-flip p,0.125
per replicate,"{""A1a"": {""delta"": 0.029411764705882353, ""fixed..."
guardrails pass,True
engineering criterion (+0.05 in average and each replicate),False
broken > fixed/2,False
A1 - F8 > 0,True


,delta,ci95,p05_one_sided,sign_flip_p
F8 - B1s,0.019608,"[-0.047625, 0.078954]",-0.035729,0.328125
A1 - F8,0.004902,"[-0.050011, 0.064516]",-0.041667,0.5
F8 - B1s on triggered,0.060976,"[0.0, 0.125]",0.016667,0.125
F8 - B1s on untriggered,-0.008197,"[-0.103448, 0.078958]",-0.087719,0.625


,B1s correct,frozen correct,flips
B1s vs frozen L-B1,44,46,6
B1s vs frozen L-B1r,44,46,8


,B1s,A1 (avg),F8 (avg),A1 - B1s,F8 - B1s,n
pending as wrong,44.0,46.5,46.5,0.0248,0.0248,101.0
pending as correct,51.0,53.0,49.5,0.0198,-0.0149,101.0


Frozen E9 evaluation reproduced exactly (B1s 44, F8a 46, F8b 46, A1a 47, A1b 46; no_meaningful_gain).
E9 evaluated from experiments/frozen/e9/run_20261125T171152006205Z_57d16dea_full. S-v1 verdict: no_meaningful_gain. New blind S-v2 items for the owner: 13 (data/processed/mmlongbench_doc/e9/adjudication_queue.csv).


## Results and interpretation (E9, development data)

**Verdict under the frozen rules (S-v1): `no_meaningful_gain`.** B1 stays as it is: ColQwen2 top 4, one read.

| | B1s | F8a | F8b | A1a | A1b |
|---|---:|---:|---:|---:|---:|
| S-v1 correct / 102 | 44 | 46 | 46 | 47 | 46 |
| Correct abstentions / 16 | 9 | 9 | 8 | 9 | 9 |
| Mean prompt tokens | 3,977 | 7,641 | 7,641 | 5,421 | 5,421 |

| Contrast (S-v1) | Δ | 95 % CI | Sign-flip p |
|---|---:|---|---:|
| **A1 − B1s (primary)** | **+0.025** | [0.000, +0.051] | 0.125 |
| F8 − B1s | +0.020 | [−0.048, +0.079] | 0.33 |
| A1 − F8 | +0.005 | [−0.050, +0.065] | 0.50 |

Per replicate, A1 fixed 3 questions and broke none (a), then fixed 2 and broke none (b). Every guardrail passes. The engineering criterion (+0.05 on average and in each replicate) does not.

**What the evidence says, beside the numbers above.**
1. **The trigger points in the right direction, but there is little to recover.**
   - Reading 8 pages helps on triggered questions (+0.061) and slightly hurts the rest (−0.008). That is the signature of a useful trigger.
   - A1 never broke a correct answer in 204 derived replies.
   - The gain is about 2.5 questions, which is what the independent Stage 0 audit predicted before any paid call. U3 detects *incomplete* evidence well (AUROC 0.78), but it is at chance for the questions that 8 pages can *complete* (AUROC 0.48). This caps A1 − B1s at about +0.069.
2. **Always reading 8 pages is not better than 4 for this reader.** F8 − B1s = +0.020, and its interval spans zero. The 4-page cap, inherited from GPT-4o-mini's image billing, costs no measurable accuracy with `gpt-6-luna`. It also halves prompt tokens.
3. **Session drift is real and is why B1s exists.** On byte-identical requests sent the same day, B1s scored 44 against the frozen E8 L-B1's 46 (6 flips; 8 against L-B1r). Judged against the frozen replies, A1's gain would shrink to +0.005. A future claim of a gain under about 7 questions on these 102 needs a same-session control and replicates.
4. **Scoring does not change the conclusion.** The 13 new gray items are still unlabelled. Over every consistent labelling of them, S-v2 puts A1 − B1s between +0.005 and +0.040 (+0.042 if "unsure" is allowed). No labelling reaches ±0.05 (independent audit, `experiments/frozen/e9/AUDIT.json`).
5. **Audit caveats.**
   - The provider's prompt cache was active in this session but not in E8. The replies were generated fresh, and replicate agreement did not rise.
   - The system fingerprint was absent from every call, so that guard could never fire.
   - Only 3 documents changed at all, so the smallest attainable sign-flip p was 0.125.

**Limitations.**
- 102 answerable questions in 14 development documents, already used to design B1.
- One trigger family (retrieval scores) and one deeper read (ranks 5–8).
- Evidence annotations come from a single annotator.
- The reader alias cannot be pinned.

A larger sample could resolve a +2.5-point effect. That is a reason to report A1 as a pre-registered *secondary* comparison in the final held-out evaluation, if the owner wants it, not a reason to adopt it.

**Cost.** $0.151647 of the $0.50 ceiling (6 smoke calls + 354 calls, all first-attempt successes).